# Goal 4 — Human perceptual correspondence  
## Final family-based signal–human analysis

This notebook tests whether signal-derived recording-quality indicators correspond to construct-matched human perceptual families.

The analysis is deliberately **family-based in the main manuscript**:

- the 516 production recordings contribute one human family rating each;
- rater identity is treated as a nuisance adjustment, not a scientific outcome;
- the 69 four-rater recordings provide reliability context only and are not pooled with production outcomes;
- primary models use family-level presence for QADD, QGAIN, QREV, and QCHAN;
- individual signal indicators remain separate because no family score is constructed;
- narrow item-level comparisons are secondary refinements only where the item has a direct physical match;
- cross-family models evaluate specificity: matched indicators should correspond more strongly to their intended perceptual family than to unrelated families;
- QDIST and QTEMP remain descriptive because machine detections are sparse and their positive human labels are weakly reproducible or rare.

The intended manuscript conclusion is not that human ratings reproduce the signal measurements. It is that perceptual correspondence is selective, family-dependent, and complementary to analytical signal characterization.


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import platform
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.cov_struct import Exchangeable, Independence
from statsmodels.stats.multitest import multipletests
from IPython.display import display

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

pd.set_option("display.max_columns", 240)
pd.set_option("display.max_rows", 240)
pd.set_option("display.width", 260)

# ------------------------------------------------------------------
# Run configuration
# ------------------------------------------------------------------
FINAL_RUN = True
SEED = 20260805

N_SUBJECT_COLLAPSED_RESAMPLES = 1000 if FINAL_RUN else 100
MIN_RESAMPLE_SUCCESS = 0.90

MIN_MODEL_RECORDINGS = 50
MIN_HUMAN_POSITIVES = 20
MIN_HUMAN_NEGATIVES = 20
MIN_POSITIVE_PARTICIPANTS = 10
MIN_NEGATIVE_PARTICIPANTS = 10
MIN_POSITIVE_RATERS = 2
MIN_PREDICTOR_UNIQUE = 2

EXPECTED_FROZEN_RECORDINGS = 519
EXPECTED_FROZEN_PARTICIPANTS = 224
EXPECTED_HUMAN_RECORDINGS = 516

EXPECTED_RATERS = ["Abbas", "Liya", "Samaana", "Samara"]

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
def locate_analysis_root() -> Path:
    configured = os.environ.get("PAPER1_ANALYSIS_ROOT", "").strip()
    if configured:
        return Path(configured).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the analysis repository root")

ANALYSIS_ROOT = locate_analysis_root()
DEFAULT_PAPER_ROOT = ANALYSIS_ROOT.parent
PAPER_ROOT = Path(os.environ.get("PAPER1_ROOT", DEFAULT_PAPER_ROOT))
PIPELINE_ROOT = Path(os.environ.get("PAPER1_PIPELINE_ROOT", PAPER_ROOT / "Paper_1_Final_Cleanup"))

DATA_FREEZE_DIR = Path(
    os.environ.get(
        "PAPER1_DATA_FREEZE_DIR",
        PIPELINE_ROOT / "outputs" / "data",
    )
)
FAMILY_FREEZE_DIR = Path(
    os.environ.get(
        "PAPER1_FAMILY_FREEZE_DIR",
        PIPELINE_ROOT / "outputs" / "features",
    )
)

STAGE1_ROOT = (
    ANALYSIS_ROOT
    / "outputs"
    / "goal3_human_correspondence"
    / "stage1_annotation_audit"
)
STAGE1B_ROOT = (
    ANALYSIS_ROOT
    / "outputs"
    / "goal3_human_correspondence"
    / "stage1b_human_reference_reliability"
)

OUTPUT_ROOT = (
    ANALYSIS_ROOT
    / "outputs"
    / "goal3_human_correspondence"
    / "stage2_construct_matched_correspondence"
)
TABLE_DIR = OUTPUT_ROOT / "tables"
MODEL_DIR = OUTPUT_ROOT / "models"
LOG_DIR = OUTPUT_ROOT / "logs"
MANIFEST_DIR = OUTPUT_ROOT / "manifest"
FIGURE_MAIN_DIR = OUTPUT_ROOT / "figures" / "main"
FIGURE_SUPP_DIR = OUTPUT_ROOT / "figures" / "supplementary"
FIGURE_SOURCE_DIR = OUTPUT_ROOT / "figures" / "source_data"

for directory in [
    TABLE_DIR,
    MODEL_DIR,
    LOG_DIR,
    MANIFEST_DIR,
    FIGURE_MAIN_DIR,
    FIGURE_SUPP_DIR,
    FIGURE_SOURCE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

FROZEN_SPEC = {
    "analysis_name": "Construct-matched human perceptual correspondence",
    "human_reliability_subset": "69 recordings rated by all four RAs",
    "production_human_reference": "one RA rating per recording",
    "expected_production_recordings": EXPECTED_HUMAN_RECORDINGS,
    "primary_model": (
        "binomial GEE with exchangeable participant correlation, "
        "rater and cohort fixed effects, and centered log recording duration"
    ),
    "continuous_effect_scale": "one frozen-population transformed IQR",
    "qchan_directional_effect_scale": "positive state versus zero state",
    "multiplicity": (
        "Benjamini-Hochberg FDR separately for primary family-level and "
        "secondary item-level construct maps"
    ),
    "subject_collapsed_resamples": N_SUBJECT_COLLAPSED_RESAMPLES,
    "primary_human_interval_union_sec": 0.0,
    "specificity_analysis": (
        "adjusted feature-by-family association matrix with exact family-label permutation"
    ),
    "qdist_qtemp_role": "descriptive only",
    "no_imputation": True,
    "no_global_score": True,
    "final_run": FINAL_RUN,
    "seed": SEED,
}

(
    MANIFEST_DIR / "frozen_analysis_spec.json"
).write_text(
    json.dumps(FROZEN_SPEC, indent=2),
    encoding="utf-8",
)

print(json.dumps(FROZEN_SPEC, indent=2))
print("DATA_FREEZE_DIR:", DATA_FREEZE_DIR)
print("FAMILY_FREEZE_DIR:", FAMILY_FREEZE_DIR)
print("STAGE1_ROOT:", STAGE1_ROOT)
print("STAGE1B_ROOT:", STAGE1B_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## 1. Shared utilities

In [ ]:
def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.casefold()

    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)

    if suffix in {".parquet", ".pq"}:
        try:
            return pd.read_parquet(path)
        except ImportError:
            csv_fallback = path.with_suffix(".csv")

            if csv_fallback.exists():
                warnings.warn(
                    f"Parquet engine unavailable; using {csv_fallback}",
                    RuntimeWarning,
                )
                return pd.read_csv(
                    csv_fallback,
                    low_memory=False,
                )

            raise

    raise ValueError(f"Unsupported table type: {path}")


def resolve_table(
    root: Path,
    exact_relatives: Sequence[str],
    patterns: Sequence[str],
    *,
    label: str,
) -> Path:
    for relative in exact_relatives:
        candidate = root / relative

        if candidate.exists():
            return candidate

    matches: list[Path] = []

    for pattern in patterns:
        matches.extend(
            path
            for path in root.rglob(pattern)
            if path.is_file()
        )

    matches = sorted(
        {path.resolve() for path in matches},
        key=lambda path: (
            0 if path.suffix.casefold() in {".parquet", ".pq"} else 1,
            "checkpoint" in str(path).casefold(),
            "source_candidate" in str(path).casefold(),
            len(str(path)),
            str(path).casefold(),
        ),
    )

    matches = [
        path
        for path in matches
        if "checkpoint" not in str(path).casefold()
    ] or matches

    if not matches:
        raise FileNotFoundError(
            f"Could not locate {label} below {root}. "
            f"Patterns: {patterns}"
        )

    return matches[0]


def canonical_recording_id(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""

    text = str(value).strip().replace("\\", "/")
    name = text.rsplit("/", 1)[-1]
    lower_name = name.casefold()

    for suffix in [
        "_segments.csv",
        ".segments.csv",
        ".csv",
        ".wav",
        ".mp3",
        ".m4a",
        ".mp4",
        ".flac",
    ]:
        if lower_name.endswith(suffix):
            name = name[: -len(suffix)]
            break

    return re.sub(r"\s+", "", name).casefold()


def canonical_participant_id(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""

    text = str(value).strip()

    if text.casefold() in {
        "",
        "nan",
        "none",
        "null",
        "na",
    }:
        return ""

    return re.sub(r"\s+", "", text).casefold()


def safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def boolean_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)

    return (
        series.astype("string")
        .str.strip()
        .str.casefold()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
        .fillna(False)
        .astype(bool)
    )


def transform_feature(
    series: pd.Series,
    transform: str,
) -> pd.Series:
    values = safe_numeric(series)

    if transform == "none":
        return values

    if transform == "log1p":
        if (values.dropna() < 0).any():
            raise ValueError(
                "log1p requested for a feature containing negative values."
            )

        return np.log1p(values)

    if transform == "asinh":
        return np.arcsinh(values)

    raise ValueError(f"Unknown transform: {transform}")


def robust_scale(
    series: pd.Series,
) -> tuple[pd.Series, float, float]:
    values = safe_numeric(series)
    median = float(values.median(skipna=True))
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = float(q3 - q1)

    if not np.isfinite(iqr) or iqr <= 0:
        return (
            pd.Series(np.nan, index=series.index, dtype=float),
            median,
            iqr,
        )

    return (
        (values - median) / iqr,
        median,
        iqr,
    )


def bh_adjust(
    frame: pd.DataFrame,
    *,
    group_column: str,
    p_column: str = "p_value",
) -> pd.DataFrame:
    output = frame.copy()
    output["q_value"] = np.nan

    for _, group in output.groupby(
        group_column,
        dropna=False,
    ):
        mask = (
            group[p_column].notna()
            & np.isfinite(group[p_column])
        )

        if mask.any():
            indices = group.index[mask]

            output.loc[
                indices,
                "q_value",
            ] = multipletests(
                output.loc[
                    indices,
                    p_column,
                ].astype(float),
                method="fdr_bh",
            )[1]

    return output


def export_dataframe(
    frame: pd.DataFrame,
    path: Path,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def save_figure(
    figure: mpl.figure.Figure,
    stem: str,
    *,
    main: bool,
) -> dict[str, str]:
    directory = (
        FIGURE_MAIN_DIR
        if main
        else FIGURE_SUPP_DIR
    )

    paths = {
        "png": directory / f"{stem}.png",
        "svg": directory / f"{stem}.svg",
    }

    figure.savefig(
        paths["png"],
        dpi=600 if main else 350,
        facecolor="white",
        bbox_inches="tight",
    )
    figure.savefig(
        paths["svg"],
        facecolor="white",
        bbox_inches="tight",
    )

    if main:
        paths["pdf"] = directory / f"{stem}.pdf"
        figure.savefig(
            paths["pdf"],
            facecolor="white",
            bbox_inches="tight",
        )

    return {
        key: str(value)
        for key, value in paths.items()
    }


## 2. Load the Stage-1 human tables and Stage-1B reliability outputs

The final production analysis uses recording-level presence only. Human duration fractions are not modeled because several families were commonly marked for the full recording and duration agreement was not uniformly reproducible.


In [ ]:
stage1_paths = {
    # All annotation ratings, including the shared four-rater reliability subset.
    "family_all": (
        STAGE1_ROOT
        / "tables"
        / "humanqc_recording_signal_family_summary.csv"
    ),
    "item_all": (
        STAGE1_ROOT
        / "tables"
        / "humanqc_recording_item_summary.csv"
    ),
    # Frozen-population production annotations: one rater per recording.
    "family": (
        STAGE1_ROOT
        / "tables"
        / "humanqc_recording_signal_family_analysis_population.csv"
    ),
    "source_family": (
        STAGE1_ROOT
        / "tables"
        / "humanqc_recording_source_family_analysis_population.csv"
    ),
    "item": (
        STAGE1_ROOT
        / "tables"
        / "humanqc_recording_item_analysis_population.csv"
    ),
    "population_merge": (
        STAGE1_ROOT
        / "tables"
        / "humanqc_population_merge_summary.csv"
    ),
    "duplicate_audit": (
        STAGE1_ROOT
        / "tables"
        / "humanqc_duplicate_recording_audit.csv"
    ),
    "population_metadata": (
        STAGE1_ROOT
        / "tables"
        / "frozen_519_population_with_metadata.csv"
    ),
}

stage1b_paths = {
    "overall": (
        STAGE1B_ROOT
        / "tables"
        / "human_reference_overall_reliability.csv"
    ),
    "family": (
        STAGE1B_ROOT
        / "tables"
        / "human_reference_family_presence_reliability.csv"
    ),
    "item": (
        STAGE1B_ROOT
        / "tables"
        / "human_reference_item_presence_reliability.csv"
    ),
    "inventory": (
        STAGE1B_ROOT
        / "tables"
        / "human_reference_complete_four_rater_inventory.csv"
    ),
    "temporal": (
        STAGE1B_ROOT
        / "tables"
        / "human_reference_family_temporal_agreement.csv"
    ),
}

required_human_paths = {
    **stage1_paths,
    **{
        f"stage1b_{key}": path
        for key, path in stage1b_paths.items()
    },
}

missing_human_paths = [
    str(path)
    for path in required_human_paths.values()
    if not path.exists()
]

if missing_human_paths:
    raise FileNotFoundError(
        "Required human-reference outputs are missing:\n"
        + "\n".join(missing_human_paths)
    )

human_family_all = read_table(
    stage1_paths["family_all"]
)
human_item_all = read_table(
    stage1_paths["item_all"]
)

human_family = read_table(
    stage1_paths["family"]
)
human_source_family = read_table(
    stage1_paths["source_family"]
)
human_item = read_table(
    stage1_paths["item"]
)
population_merge = read_table(
    stage1_paths["population_merge"]
)
duplicate_audit = read_table(
    stage1_paths["duplicate_audit"]
)
population_metadata = read_table(
    stage1_paths["population_metadata"]
)

reliability_overall = read_table(
    stage1b_paths["overall"]
)
reliability_family = read_table(
    stage1b_paths["family"]
)
reliability_item = read_table(
    stage1b_paths["item"]
)
reliability_inventory = read_table(
    stage1b_paths["inventory"]
)
reliability_temporal = read_table(
    stage1b_paths["temporal"]
)

print(
    "Production human recordings:",
    human_family["recording_id"].nunique(),
)
print(
    "All annotation recordings:",
    human_family_all["recording_id"].nunique(),
)
print(
    "Complete four-rater reliability recordings:",
    len(reliability_inventory),
)
display(reliability_overall)


## 3. Summarize human-label reliability without making it the primary result

Only the shared 69-recording subset is used here. Reliability supplies an interpretation boundary for the later correspondence models; rater-specific prevalence and agreement remain supplementary. A strict majority of four raters is defined as at least three.


In [ ]:
def build_reliability_matrix(
    frame: pd.DataFrame,
    outcome_column: str,
    outcome_value: str,
) -> pd.DataFrame:
    complete_ids = reliability_inventory[
        "recording_id"
    ].tolist()

    subset = frame.loc[
        frame["recording_id"].isin(complete_ids)
        & frame[outcome_column].eq(outcome_value)
    ].copy()

    matrix = subset.pivot(
        index="recording_id",
        columns="rater",
        values="present",
    ).reindex(
        index=complete_ids,
        columns=EXPECTED_RATERS,
    )

    if matrix.isna().any().any():
        raise RuntimeError(
            f"Incomplete reliability matrix for {outcome_value}."
        )

    return matrix.astype(bool)


strict_consensus_rows = []

for family in [
    "QADD",
    "QGAIN",
    "QREV",
    "QCHAN",
    "QDIST",
    "QTEMP",
]:
    matrix = build_reliability_matrix(
        human_family_all,
        "signal_family",
        family,
    )

    positive_count = matrix.sum(axis=1)

    strict_consensus_rows.append(
        {
            "signal_family": family,
            "any_positive_rater": int(
                positive_count.ge(1).sum()
            ),
            "at_least_two_raters": int(
                positive_count.ge(2).sum()
            ),
            "strict_majority_at_least_three": int(
                positive_count.ge(3).sum()
            ),
            "all_four_raters": int(
                positive_count.eq(4).sum()
            ),
            "exactly_two_raters": int(
                positive_count.eq(2).sum()
            ),
            "exactly_one_rater": int(
                positive_count.eq(1).sum()
            ),
        }
    )

strict_consensus = pd.DataFrame(
    strict_consensus_rows
)

human_reliability_context = (
    reliability_family.loc[
        reliability_family[
            "signal_family"
        ].isin(
            [
                "QADD",
                "QGAIN",
                "QREV",
                "QCHAN",
                "QDIST",
                "QTEMP",
            ]
        )
    ]
    .merge(
        strict_consensus,
        on="signal_family",
        how="inner",
        validate="one_to_one",
    )
)

human_reliability_context[
    "interpretation"
] = np.select(
    [
        human_reliability_context[
            "mean_pairwise_positive_agreement"
        ].ge(0.50),
        human_reliability_context[
            "mean_pairwise_positive_agreement"
        ].ge(0.25),
    ],
    [
        "positive labels reproducible",
        "positive labels moderately reproducible",
    ],
    default=(
        "positive labels weakly reproducible or sparse"
    ),
)

display(
    human_reliability_context[
        [
            "signal_family",
            "observed_agreement",
            "fleiss_kappa",
            "gwet_ac1",
            "mean_pairwise_positive_agreement",
            "mean_pairwise_negative_agreement",
            "strict_majority_at_least_three",
            "cochran_q_q",
            "interpretation",
        ]
    ]
)

export_dataframe(
    strict_consensus,
    TABLE_DIR / "corrected_four_rater_consensus_counts.csv",
)
export_dataframe(
    human_reliability_context,
    TABLE_DIR / "human_reliability_context.csv",
)


## 4. Discover and hash the frozen signal inputs

In [ ]:
metadata_path = stage1_paths["population_metadata"]

family_paths = {
    "QADD": resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r"families\02_QADD\features.parquet",
            r"additive_interference\qadd-v4.2.0\tables\qadd_v420_recording_features.parquet",
            r"additive_interference\qadd-v4.2.0\tables\qadd_v420_recording_features.csv",
        ],
        patterns=[
            "qadd_v420_recording_features.parquet",
            "qadd_v420_recording_features.csv",
        ],
        label="QADD recording features",
    ),
    "QGAIN": resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r"families\01_QGAIN\features.parquet",
            r"gain_dynamics\qgain-v4.1.0\tables\qgain_v410_recording_features.parquet",
            r"gain_dynamics\qgain-v4.1.0\tables\qgain_v410_recording_features.csv",
        ],
        patterns=[
            "qgain_v410_recording_features.parquet",
            "qgain_v410_recording_features.csv",
        ],
        label="QGAIN recording features",
    ),
    "QREV": resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r"families\03_QREV\features.parquet",
            r"reverberation\qrev-v4.0.0\tables\qrev_v400_analysis_features.parquet",
            r"reverberation\qrev-v4.0.0\tables\qrev_v400_analysis_features.csv",
        ],
        patterns=[
            "qrev_v400_analysis_features.parquet",
            "qrev_v400_analysis_features.csv",
        ],
        label="QREV analysis features",
    ),
    "QCHAN": resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r"families\04_QCHAN\features.parquet",
            r"channel_device\qchan-v4.0.0\tables\qchan_v400_analysis_features.parquet",
            r"channel_device\qchan-v4.0.0\tables\qchan_v400_analysis_features.csv",
        ],
        patterns=[
            "qchan_v400_analysis_features.parquet",
            "qchan_v400_analysis_features.csv",
        ],
        label="QCHAN analysis features",
    ),
    "QDIST": resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            "families/05_QDIST/features.parquet",
            "nonlinear_distortion/qdist-v4.1.0/finalization/tables/qdist_v410_analysis_features.parquet",
            "nonlinear_distortion/qdist-v4.1.0/finalization/tables/qdist_v410_analysis_features.csv",
            "nonlinear_distortion/qdist-v4.1.0/tables/qdist_v410_analysis_features.parquet",
            "nonlinear_distortion/qdist-v4.1.0/tables/qdist_v410_analysis_features.csv",
        ],
        patterns=[
            "qdist_v410_analysis_features.parquet",
            "qdist_v410_analysis_features.csv",
        ],
        label="QDIST analysis features",
    ),
    "QTEMP": resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r"experimental\QTEMP_LEGACY_NO_RETAINED\qtemp_v100_exploratory_features.csv",
            r"temporal_discontinuity\qtemp-v1.0.0-analytical-final-no-retained\tables\qtemp_v100_exploratory_features.csv",
        ],
        patterns=[
            "qtemp_v100_exploratory_features.csv",
        ],
        label="QTEMP exploratory features",
    ),
}

input_inventory = pd.DataFrame(
    [
        {
            "input": "metadata",
            "path": str(metadata_path),
            "sha256": sha256_file(metadata_path),
            "bytes": metadata_path.stat().st_size,
        },
        *[
            {
                "input": family,
                "path": str(path),
                "sha256": sha256_file(path),
                "bytes": path.stat().st_size,
            }
            for family, path in family_paths.items()
        ],
        *[
            {
                "input": name,
                "path": str(path),
                "sha256": sha256_file(path),
                "bytes": path.stat().st_size,
            }
            for name, path in required_human_paths.items()
        ],
    ]
)

display(input_inventory)

export_dataframe(
    input_inventory,
    MANIFEST_DIR / "input_inventory.csv",
)

metadata_raw = read_table(metadata_path)
family_raw = {
    family: read_table(path)
    for family, path in family_paths.items()
}


## 5. Feature registry and frozen construct map

In [ ]:
FEATURE_REGISTRY = pd.DataFrame(
    [
        # QADD
        {
            "feature": "qadd_pause_ac_level_dbfs_median",
            "family": "QADD",
            "label": "Pause-region level",
            "unit": "dBFS",
            "transform": "none",
            "predictor_type": "continuous",
        },
        {
            "feature": "qadd_pause_level_iqr_db",
            "family": "QADD",
            "label": "Pause-level IQR",
            "unit": "dB",
            "transform": "log1p",
            "predictor_type": "continuous",
        },
        {
            "feature": "qadd_speech_pause_level_contrast_db",
            "family": "QADD",
            "label": "Speech–pause contrast",
            "unit": "dB",
            "transform": "none",
            "predictor_type": "continuous",
        },
        {
            "feature": "qadd_mains_hum_comb_score_db",
            "family": "QADD",
            "label": "Hum-comb prominence",
            "unit": "dB",
            "transform": "asinh",
            "predictor_type": "continuous",
        },
        # QGAIN
        {
            "feature": "qgain_within_segment_iqr_db",
            "family": "QGAIN",
            "label": "Within-segment IQR",
            "unit": "dB",
            "transform": "log1p",
            "predictor_type": "continuous",
        },
        {
            "feature": "qgain_between_segment_mad_db",
            "family": "QGAIN",
            "label": "Between-segment MAD",
            "unit": "dB",
            "transform": "log1p",
            "predictor_type": "continuous",
        },
        {
            "feature": "qgain_abs_drift_db_per_min",
            "family": "QGAIN",
            "label": "Absolute level drift",
            "unit": "dB/min",
            "transform": "log1p",
            "predictor_type": "continuous",
        },
        # QREV
        {
            "feature": "qrev_tail_excess_100ms_db",
            "family": "QREV",
            "label": "Early tail excess",
            "unit": "dB",
            "transform": "asinh",
            "predictor_type": "continuous",
        },
        {
            "feature": "qrev_tail_persistence_median_sec",
            "family": "QREV",
            "label": "Tail persistence",
            "unit": "s",
            "transform": "log1p",
            "predictor_type": "continuous",
        },
        {
            "feature": "qrev_srmr_norm",
            "family": "QREV",
            "label": "Normalized-fast SRMR",
            "unit": "ratio",
            "transform": "log1p",
            "predictor_type": "continuous",
        },
        # QCHAN
        {
            "feature": "qchan_ltas_distance_db",
            "family": "QCHAN",
            "label": "LTAS distance",
            "unit": "dB RMS",
            "transform": "log1p",
            "predictor_type": "continuous",
        },
        {
            "feature": "qchan_rolloff95_deficit_hz",
            "family": "QCHAN",
            "label": "Rolloff-95 deficit present",
            "unit": "state",
            "transform": "log1p",
            "predictor_type": "binary_state",
        },
        {
            "feature": "qchan_highband_ratio_deficit",
            "family": "QCHAN",
            "label": "High-band deficit present",
            "unit": "state",
            "transform": "log1p",
            "predictor_type": "binary_state",
        },
        {
            "feature": "qchan_tilt_steepening_db_per_oct",
            "family": "QCHAN",
            "label": "Tilt steepening present",
            "unit": "state",
            "transform": "log1p",
            "predictor_type": "binary_state",
        },
        # QDIST / QTEMP descriptive
        {
            "feature": "qdist_hard_clipped_sample_fraction",
            "family": "QDIST",
            "label": "Hard-clipped sample fraction",
            "unit": "fraction",
            "transform": "log1p",
            "predictor_type": "sparse_state",
        },
        {
            "feature": "qdist_hard_clip_event_rate_per_min",
            "family": "QDIST",
            "label": "Hard-clipping event rate",
            "unit": "events/min",
            "transform": "log1p",
            "predictor_type": "sparse_state",
        },
        {
            "feature": "qtemp_dropout_duration_fraction",
            "family": "QTEMP",
            "label": "Dropout-like duration fraction",
            "unit": "fraction",
            "transform": "log1p",
            "predictor_type": "sparse_state",
        },
        {
            "feature": "qtemp_dropout_event_rate_per_min",
            "family": "QTEMP",
            "label": "Dropout-like event rate",
            "unit": "events/min",
            "transform": "log1p",
            "predictor_type": "sparse_state",
        },
        {
            "feature": "qtemp_frozen_audio_duration_fraction",
            "family": "QTEMP",
            "label": "Frozen-audio duration fraction",
            "unit": "fraction",
            "transform": "log1p",
            "predictor_type": "sparse_state",
        },
    ]
)

# expected_sign:
# +1 means a larger indicator is expected with the percept;
# -1 means a smaller indicator is expected;
# 0 means no directional hypothesis is used.
#
# Primary family-level models answer the manuscript question.
# Secondary item-level models refine direct subconstruct matches only.
CONSTRUCT_MAP = pd.DataFrame(
    [
        # ----------------------------------------------------------
        # Primary family-level construct map
        # ----------------------------------------------------------
        {
            "pair_id": "FAM_QADD_01",
            "family": "QADD",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QADD",
            "human_label": "Additive interference",
            "feature": "qadd_pause_ac_level_dbfs_median",
            "expected_sign": 1,
            "relationship_role": "primary family match",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QADD_02",
            "family": "QADD",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QADD",
            "human_label": "Additive interference",
            "feature": "qadd_pause_level_iqr_db",
            "expected_sign": 1,
            "relationship_role": "supporting family match",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QADD_03",
            "family": "QADD",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QADD",
            "human_label": "Additive interference",
            "feature": "qadd_speech_pause_level_contrast_db",
            "expected_sign": -1,
            "relationship_role": "structurally related family match",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QGAIN_01",
            "family": "QGAIN",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QGAIN",
            "human_label": "Level instability",
            "feature": "qgain_within_segment_iqr_db",
            "expected_sign": 1,
            "relationship_role": "primary family match",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QGAIN_02",
            "family": "QGAIN",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QGAIN",
            "human_label": "Level instability",
            "feature": "qgain_between_segment_mad_db",
            "expected_sign": 1,
            "relationship_role": "supporting family match",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QGAIN_03",
            "family": "QGAIN",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QGAIN",
            "human_label": "Level instability",
            "feature": "qgain_abs_drift_db_per_min",
            "expected_sign": 1,
            "relationship_role": "exploratory family match",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QREV_01",
            "family": "QREV",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QREV",
            "human_label": "Reverberation or echo",
            "feature": "qrev_srmr_norm",
            "expected_sign": -1,
            "relationship_role": "full-support family comparator",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QREV_02",
            "family": "QREV",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QREV",
            "human_label": "Reverberation or echo",
            "feature": "qrev_tail_excess_100ms_db",
            "expected_sign": 1,
            "relationship_role": "conditional boundary match",
            "main_display": True,
            "specificity_core": False,
        },
        {
            "pair_id": "FAM_QREV_03",
            "family": "QREV",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QREV",
            "human_label": "Reverberation or echo",
            "feature": "qrev_tail_persistence_median_sec",
            "expected_sign": 1,
            "relationship_role": "conditional non-independent match",
            "main_display": False,
            "specificity_core": False,
        },
        {
            "pair_id": "FAM_QCHAN_01",
            "family": "QCHAN",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QCHAN",
            "human_label": "Platform effects",
            "feature": "qchan_ltas_distance_db",
            "expected_sign": 1,
            "relationship_role": "broad family match",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QCHAN_02",
            "family": "QCHAN",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QCHAN",
            "human_label": "Platform effects",
            "feature": "qchan_rolloff95_deficit_hz",
            "expected_sign": 1,
            "relationship_role": "directional family state",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QCHAN_03",
            "family": "QCHAN",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QCHAN",
            "human_label": "Platform effects",
            "feature": "qchan_highband_ratio_deficit",
            "expected_sign": 1,
            "relationship_role": "supporting directional state",
            "main_display": True,
            "specificity_core": True,
        },
        {
            "pair_id": "FAM_QCHAN_04",
            "family": "QCHAN",
            "inference_tier": "primary_family",
            "human_level": "signal_family",
            "human_value": "QCHAN",
            "human_label": "Platform effects",
            "feature": "qchan_tilt_steepening_db_per_oct",
            "expected_sign": 1,
            "relationship_role": "phenotype-sensitive state",
            "main_display": False,
            "specificity_core": False,
        },

        # ----------------------------------------------------------
        # Secondary item-level refinements
        # ----------------------------------------------------------
        {
            "pair_id": "ITEM_QADD_HVAC",
            "family": "QADD",
            "inference_tier": "secondary_item",
            "human_level": "item",
            "human_value": "HVAC",
            "human_label": "HVAC",
            "feature": "qadd_mains_hum_comb_score_db",
            "expected_sign": 1,
            "relationship_role": "narrow targeted item match",
            "main_display": False,
            "specificity_core": False,
        },
        {
            "pair_id": "ITEM_QGAIN_01",
            "family": "QGAIN",
            "inference_tier": "secondary_item",
            "human_level": "item",
            "human_value": "Volume changes",
            "human_label": "Volume changes",
            "feature": "qgain_within_segment_iqr_db",
            "expected_sign": 1,
            "relationship_role": "item-level refinement",
            "main_display": False,
            "specificity_core": False,
        },
        {
            "pair_id": "ITEM_QGAIN_02",
            "family": "QGAIN",
            "inference_tier": "secondary_item",
            "human_level": "item",
            "human_value": "Volume changes",
            "human_label": "Volume changes",
            "feature": "qgain_between_segment_mad_db",
            "expected_sign": 1,
            "relationship_role": "item-level refinement",
            "main_display": False,
            "specificity_core": False,
        },
        {
            "pair_id": "ITEM_QGAIN_03",
            "family": "QGAIN",
            "inference_tier": "secondary_item",
            "human_level": "item",
            "human_value": "Volume changes",
            "human_label": "Volume changes",
            "feature": "qgain_abs_drift_db_per_min",
            "expected_sign": 1,
            "relationship_role": "item-level refinement",
            "main_display": False,
            "specificity_core": False,
        },
        {
            "pair_id": "ITEM_QCHAN_01",
            "family": "QCHAN",
            "inference_tier": "secondary_item",
            "human_level": "item",
            "human_value": "Muffled, filtered, telephone-like",
            "human_label": "Muffled or filtered",
            "feature": "qchan_ltas_distance_db",
            "expected_sign": 1,
            "relationship_role": "item-level refinement",
            "main_display": False,
            "specificity_core": False,
        },
        {
            "pair_id": "ITEM_QCHAN_02",
            "family": "QCHAN",
            "inference_tier": "secondary_item",
            "human_level": "item",
            "human_value": "Muffled, filtered, telephone-like",
            "human_label": "Muffled or filtered",
            "feature": "qchan_rolloff95_deficit_hz",
            "expected_sign": 1,
            "relationship_role": "item-level refinement",
            "main_display": False,
            "specificity_core": False,
        },
        {
            "pair_id": "ITEM_QCHAN_03",
            "family": "QCHAN",
            "inference_tier": "secondary_item",
            "human_level": "item",
            "human_value": "Muffled, filtered, telephone-like",
            "human_label": "Muffled or filtered",
            "feature": "qchan_highband_ratio_deficit",
            "expected_sign": 1,
            "relationship_role": "item-level refinement",
            "main_display": False,
            "specificity_core": False,
        },
        {
            "pair_id": "ITEM_QCHAN_04",
            "family": "QCHAN",
            "inference_tier": "secondary_item",
            "human_level": "item",
            "human_value": "Muffled, filtered, telephone-like",
            "human_label": "Muffled or filtered",
            "feature": "qchan_tilt_steepening_db_per_oct",
            "expected_sign": 1,
            "relationship_role": "item-level refinement",
            "main_display": False,
            "specificity_core": False,
        },
    ]
)

CONSTRUCT_MAP = CONSTRUCT_MAP.merge(
    FEATURE_REGISTRY[
        [
            "feature",
            "label",
            "unit",
            "transform",
            "predictor_type",
        ]
    ].rename(
        columns={
            "label": "feature_label",
        }
    ),
    on="feature",
    how="left",
    validate="many_to_one",
)

if CONSTRUCT_MAP[
    [
        "feature_label",
        "transform",
        "predictor_type",
    ]
].isna().any().any():
    raise RuntimeError(
        "Construct map contains a feature absent from the registry."
    )

PRIMARY_FAMILY_MAP = CONSTRUCT_MAP.loc[
    CONSTRUCT_MAP[
        "inference_tier"
    ].eq("primary_family")
].copy()

SECONDARY_ITEM_MAP = CONSTRUCT_MAP.loc[
    CONSTRUCT_MAP[
        "inference_tier"
    ].eq("secondary_item")
].copy()

SPECIFICITY_FEATURE_MAP = (
    PRIMARY_FAMILY_MAP.loc[
        PRIMARY_FAMILY_MAP[
            "specificity_core"
        ]
    ][
        [
            "family",
            "feature",
            "feature_label",
            "expected_sign",
            "predictor_type",
        ]
    ]
    .drop_duplicates(
        subset=["feature"]
    )
)

display(CONSTRUCT_MAP)

export_dataframe(
    FEATURE_REGISTRY,
    TABLE_DIR / "feature_registry.csv",
)
export_dataframe(
    CONSTRUCT_MAP,
    TABLE_DIR / "frozen_construct_map.csv",
)
export_dataframe(
    SPECIFICITY_FEATURE_MAP,
    TABLE_DIR / "specificity_feature_map.csv",
)


## 6. Build the common 519-recording signal matrix

In [ ]:
def select_family_columns(
    frame: pd.DataFrame,
    family: str,
) -> pd.DataFrame:
    id_column = (
        "logical_recording_id"
        if "logical_recording_id" in frame.columns
        else "recording_id"
        if "recording_id" in frame.columns
        else None
    )

    if id_column is None:
        raise KeyError(
            f"{family}: no recording ID column was found."
        )

    selected_columns = [
        id_column,
        *[
            column
            for column in frame.columns
            if str(column).casefold().startswith(
                family.casefold()
            )
        ],
    ]

    selected = frame.loc[
        :,
        list(dict.fromkeys(selected_columns)),
    ].copy()

    selected["recording_id"] = selected[
        id_column
    ].map(canonical_recording_id)

    if id_column != "recording_id":
        selected = selected.drop(columns=id_column)

    if selected["recording_id"].duplicated().any():
        examples = selected.loc[
            selected["recording_id"].duplicated(
                keep=False
            ),
            "recording_id",
        ].head(10)

        raise RuntimeError(
            f"{family}: duplicate recording IDs: {examples.tolist()}"
        )

    return selected


family_tables = {
    family: select_family_columns(
        frame,
        family,
    )
    for family, frame in family_raw.items()
}

id_sets = {
    family: set(
        frame["recording_id"].dropna()
    )
    for family, frame in family_tables.items()
}

intersection_ids = set.intersection(
    *id_sets.values()
)
union_ids = set.union(
    *id_sets.values()
)

if (
    len(intersection_ids)
    != EXPECTED_FROZEN_RECORDINGS
    or len(union_ids)
    != EXPECTED_FROZEN_RECORDINGS
):
    raise RuntimeError(
        "The six family tables do not define one common "
        f"{EXPECTED_FROZEN_RECORDINGS}-recording population. "
        f"Intersection={len(intersection_ids)}, union={len(union_ids)}."
    )

signal_matrix = pd.DataFrame(
    {
        "recording_id": sorted(
            intersection_ids
        )
    }
)

for family in [
    "QADD",
    "QGAIN",
    "QREV",
    "QCHAN",
    "QDIST",
    "QTEMP",
]:
    signal_matrix = signal_matrix.merge(
        family_tables[family],
        on="recording_id",
        how="left",
        validate="one_to_one",
    )

required_features = FEATURE_REGISTRY[
    "feature"
].tolist()

missing_features = [
    feature
    for feature in required_features
    if feature not in signal_matrix.columns
]

if missing_features:
    raise KeyError(
        f"Missing required frozen feature columns: {missing_features}"
    )

metadata = metadata_raw.copy()

required_metadata_columns = {
    "recording_id",
    "participant_id",
    "cohort",
}

missing_metadata_columns = (
    required_metadata_columns
    - set(metadata.columns)
)

if missing_metadata_columns:
    raise KeyError(
        "The governed Stage-1 population table is missing columns: "
        f"{sorted(missing_metadata_columns)}"
    )

metadata["recording_id"] = metadata[
    "recording_id"
].map(canonical_recording_id)
metadata["participant_id"] = metadata[
    "participant_id"
].map(canonical_participant_id)
metadata["cohort"] = (
    metadata["cohort"]
    .astype("string")
    .str.strip()
    .str.upper()
)

metadata = metadata.loc[
    metadata["recording_id"].isin(
        intersection_ids
    )
].copy()

metadata = metadata.drop_duplicates(
    subset=["recording_id"],
    keep="first",
)

analysis_519 = metadata[
    [
        "recording_id",
        "participant_id",
        "cohort",
    ]
].merge(
    signal_matrix,
    on="recording_id",
    how="inner",
    validate="one_to_one",
)

population_audit = pd.DataFrame(
    [
        {
            "check": "Frozen recording count",
            "observed": len(analysis_519),
            "expected": EXPECTED_FROZEN_RECORDINGS,
            "passed": (
                len(analysis_519)
                == EXPECTED_FROZEN_RECORDINGS
            ),
        },
        {
            "check": "Frozen participant count",
            "observed": analysis_519[
                "participant_id"
            ].nunique(),
            "expected": EXPECTED_FROZEN_PARTICIPANTS,
            "passed": (
                analysis_519[
                    "participant_id"
                ].nunique()
                == EXPECTED_FROZEN_PARTICIPANTS
            ),
        },
        {
            "check": "Missing participant IDs",
            "observed": int(
                analysis_519[
                    "participant_id"
                ].eq("").sum()
            ),
            "expected": 0,
            "passed": not analysis_519[
                "participant_id"
            ].eq("").any(),
        },
        {
            "check": "Cohort values",
            "observed": " | ".join(
                sorted(
                    analysis_519[
                        "cohort"
                    ].dropna().unique()
                )
            ),
            "expected": "ALS | CONTROL",
            "passed": set(
                analysis_519[
                    "cohort"
                ].dropna()
            )
            == {"ALS", "CONTROL"},
        },
    ]
)

display(population_audit)

if not population_audit["passed"].all():
    raise RuntimeError(
        "Blocking frozen-population audit failed."
    )

export_dataframe(
    population_audit,
    LOG_DIR / "signal_population_audit.csv",
)


## 7. Transform and scale indicators using the full frozen population

In [ ]:
scaling_rows = []

for registry_row in FEATURE_REGISTRY.itertuples(
    index=False
):
    feature = registry_row.feature

    transformed = transform_feature(
        analysis_519[feature],
        registry_row.transform,
    )

    scaled, median, iqr = robust_scale(
        transformed
    )

    analysis_519[
        f"{feature}__transformed"
    ] = transformed
    analysis_519[
        f"{feature}__z"
    ] = scaled
    analysis_519[
        f"{feature}__positive"
    ] = (
        analysis_519[feature].notna()
        & safe_numeric(
            analysis_519[feature]
        ).gt(0)
    )

    scaling_rows.append(
        {
            "feature": feature,
            "family": registry_row.family,
            "transform": registry_row.transform,
            "predictor_type": (
                registry_row.predictor_type
            ),
            "n_available": int(
                transformed.notna().sum()
            ),
            "transformed_median": median,
            "transformed_iqr": iqr,
            "n_positive": int(
                analysis_519[
                    f"{feature}__positive"
                ].sum()
            ),
            "degenerate_transformed": not (
                np.isfinite(iqr)
                and iqr > 0
            ),
        }
    )

scaling_registry = pd.DataFrame(
    scaling_rows
)

display(scaling_registry)

export_dataframe(
    scaling_registry,
    TABLE_DIR / "scaling_registry.csv",
)


## 8. Build production human outcomes and merge to signals

In [ ]:
def extract_human_outcome(
    human_level: str,
    human_value: str,
) -> pd.DataFrame:
    if human_level == "signal_family":
        source = human_family
        column = "signal_family"
    elif human_level == "source_family":
        source = human_source_family
        column = "source_family"
    elif human_level == "item":
        source = human_item
        column = "item"
    else:
        raise ValueError(
            f"Unknown human level: {human_level}"
        )

    subset = source.loc[
        source[column].eq(human_value),
        [
            "recording_id",
            "rater",
            "recording_duration_sec",
            "present",
        ],
    ].copy()

    subset["recording_id"] = subset[
        "recording_id"
    ].map(canonical_recording_id)

    subset["rater"] = (
        subset["rater"]
        .astype("string")
        .str.strip()
    )
    subset["human_present"] = boolean_series(
        subset["present"]
    )

    subset = subset.drop(
        columns="present"
    )

    if subset["recording_id"].duplicated().any():
        raise RuntimeError(
            f"Production outcome {human_value} contains duplicate recordings."
        )

    return subset


unique_human_outcomes = CONSTRUCT_MAP[
    [
        "human_level",
        "human_value",
        "human_label",
    ]
].drop_duplicates()

human_outcome_tables = {}

for row in unique_human_outcomes.itertuples(
    index=False
):
    key = (
        row.human_level,
        row.human_value,
    )

    human_outcome_tables[key] = extract_human_outcome(
        row.human_level,
        row.human_value,
    )

production_recording_union = set.union(
    *[
        set(frame["recording_id"])
        for frame in human_outcome_tables.values()
    ]
)

if len(production_recording_union) != EXPECTED_HUMAN_RECORDINGS:
    raise RuntimeError(
        "Production human outcomes do not reproduce the expected "
        f"{EXPECTED_HUMAN_RECORDINGS} recordings."
    )

human_merge_audit = pd.DataFrame(
    [
        {
            "metric": "frozen_signal_recordings",
            "value": len(analysis_519),
        },
        {
            "metric": "production_human_recordings",
            "value": len(
                production_recording_union
            ),
        },
        {
            "metric": "frozen_recordings_without_human_annotation",
            "value": len(
                set(
                    analysis_519[
                        "recording_id"
                    ]
                )
                - production_recording_union
            ),
        },
    ]
)

display(human_merge_audit)

export_dataframe(
    human_merge_audit,
    TABLE_DIR / "human_signal_merge_audit.csv",
)


production_duration_table = (
    human_family[
        [
            "recording_id",
            "recording_duration_sec",
        ]
    ]
    .drop_duplicates(
        subset=["recording_id"]
    )
)

LOG_DURATION_CENTER = float(
    np.log(
        production_duration_table[
            "recording_duration_sec"
        ]
    ).median()
)

print(
    "Centered log-duration reference:",
    LOG_DURATION_CENTER,
)


## 9. Prepare construct-matched model datasets and support gates

In [ ]:
def prepare_pair_dataset(
    pair_row: Any,
    *,
    allowed_raters: list[str] | None = None,
) -> pd.DataFrame:
    human_key = (
        pair_row.human_level,
        pair_row.human_value,
    )

    human = human_outcome_tables[
        human_key
    ]

    data = analysis_519.merge(
        human,
        on="recording_id",
        how="inner",
        validate="one_to_one",
    )

    if allowed_raters is not None:
        data = data.loc[
            data["rater"].isin(
                allowed_raters
            )
        ].copy()

    if (
        pair_row.predictor_type
        == "continuous"
    ):
        data["machine_predictor"] = data[
            f"{pair_row.feature}__z"
        ]
        effect_scale = (
            "per transformed frozen-population IQR"
        )
    elif (
        pair_row.predictor_type
        == "binary_state"
    ):
        data["machine_predictor"] = data[
            f"{pair_row.feature}__positive"
        ].astype(float)
        effect_scale = (
            "positive state versus zero state"
        )
    else:
        raise ValueError(
            f"Unsupported inferential predictor type: "
            f"{pair_row.predictor_type}"
        )

    data[
        "log_recording_duration_centered"
    ] = (
        np.log(
            data[
                "recording_duration_sec"
            ]
        )
        - LOG_DURATION_CENTER
    )

    data = data[
        [
            "recording_id",
            "participant_id",
            "cohort",
            "rater",
            "recording_duration_sec",
            "log_recording_duration_centered",
            "human_present",
            "machine_predictor",
        ]
    ].dropna()

    data["human_present"] = data[
        "human_present"
    ].astype(int)

    data["rater"] = data[
        "rater"
    ].astype("category")
    data["cohort"] = pd.Categorical(
        data["cohort"],
        categories=[
            "CONTROL",
            "ALS",
        ],
    )

    data.attrs["effect_scale"] = (
        effect_scale
    )

    return data


support_rows = []

for pair_row in CONSTRUCT_MAP.itertuples(
    index=False
):
    data = prepare_pair_dataset(pair_row)

    positives = data.loc[
        data["human_present"].eq(1)
    ]
    negatives = data.loc[
        data["human_present"].eq(0)
    ]

    predictor_positive = int(
        data[
            "machine_predictor"
        ].gt(0).sum()
    )
    predictor_zero = int(
        data[
            "machine_predictor"
        ].eq(0).sum()
    )

    binary_state_support = (
        True
        if pair_row.predictor_type
        != "binary_state"
        else (
            predictor_positive >= 10
            and predictor_zero >= 10
        )
    )

    support_pass = bool(
        len(data) >= MIN_MODEL_RECORDINGS
        and len(positives)
        >= MIN_HUMAN_POSITIVES
        and len(negatives)
        >= MIN_HUMAN_NEGATIVES
        and positives[
            "participant_id"
        ].nunique()
        >= MIN_POSITIVE_PARTICIPANTS
        and negatives[
            "participant_id"
        ].nunique()
        >= MIN_NEGATIVE_PARTICIPANTS
        and positives[
            "rater"
        ].nunique()
        >= MIN_POSITIVE_RATERS
        and data[
            "machine_predictor"
        ].nunique()
        >= MIN_PREDICTOR_UNIQUE
        and binary_state_support
    )

    reasons = []

    if len(data) < MIN_MODEL_RECORDINGS:
        reasons.append(
            "insufficient recordings"
        )
    if len(positives) < MIN_HUMAN_POSITIVES:
        reasons.append(
            "insufficient human positives"
        )
    if len(negatives) < MIN_HUMAN_NEGATIVES:
        reasons.append(
            "insufficient human negatives"
        )
    if (
        positives[
            "participant_id"
        ].nunique()
        < MIN_POSITIVE_PARTICIPANTS
    ):
        reasons.append(
            "insufficient positive participants"
        )
    if (
        negatives[
            "participant_id"
        ].nunique()
        < MIN_NEGATIVE_PARTICIPANTS
    ):
        reasons.append(
            "insufficient negative participants"
        )
    if (
        positives[
            "rater"
        ].nunique()
        < MIN_POSITIVE_RATERS
    ):
        reasons.append(
            "human positives contributed by fewer than two raters"
        )
    if (
        data[
            "machine_predictor"
        ].nunique()
        < MIN_PREDICTOR_UNIQUE
    ):
        reasons.append(
            "degenerate machine predictor"
        )
    if not binary_state_support:
        reasons.append(
            "insufficient positive or zero machine states"
        )

    support_rows.append(
        {
            "pair_id": pair_row.pair_id,
            "family": pair_row.family,
            "inference_tier": (
                pair_row.inference_tier
            ),
            "human_label": (
                pair_row.human_label
            ),
            "feature": pair_row.feature,
            "feature_label": (
                pair_row.feature_label
            ),
            "predictor_type": (
                pair_row.predictor_type
            ),
            "effect_scale": (
                data.attrs["effect_scale"]
            ),
            "recordings": len(data),
            "participants": data[
                "participant_id"
            ].nunique(),
            "human_positives": int(
                data[
                    "human_present"
                ].sum()
            ),
            "human_negatives": int(
                (
                    1
                    - data[
                        "human_present"
                    ]
                ).sum()
            ),
            "positive_participants": (
                positives[
                    "participant_id"
                ].nunique()
            ),
            "negative_participants": (
                negatives[
                    "participant_id"
                ].nunique()
            ),
            "positive_raters": positives[
                "rater"
            ].nunique(),
            "predictor_unique": data[
                "machine_predictor"
            ].nunique(),
            "predictor_positive": (
                predictor_positive
            ),
            "predictor_zero": (
                predictor_zero
            ),
            "support_pass": support_pass,
            "support_reason": " | ".join(
                reasons
            ),
        }
    )

model_support = pd.DataFrame(
    support_rows
)

display(model_support)

export_dataframe(
    model_support,
    TABLE_DIR
    / "construct_pair_support_gates.csv",
)


## 10. Primary participant-grouped correspondence models

For continuous features, the coefficient is per one transformed frozen-population IQR. For QCHAN directional states, the coefficient compares positive versus zero state.

Models use two-sided tests. Expected direction is retained only for interpretation and for the direction-oriented robustness summaries.


In [ ]:
def fit_gee_model(
    data: pd.DataFrame,
    *,
    correlation: str,
) -> tuple[Any | None, str]:
    if correlation == "exchangeable":
        covariance = Exchangeable()
    elif correlation == "independence":
        covariance = Independence()
    else:
        raise ValueError(correlation)

    try:
        model = smf.gee(
            formula=(
                "human_present ~ machine_predictor "
                "+ C(rater) + C(cohort) "
                "+ log_recording_duration_centered"
            ),
            groups="participant_id",
            data=data,
            family=sm.families.Binomial(),
            cov_struct=covariance,
        )

        result = model.fit(
            maxiter=250
        )

        return result, ""

    except Exception as exc:
        return (
            None,
            f"{type(exc).__name__}: {exc}",
        )


def extract_machine_effect(
    result: Any,
) -> dict[str, float]:
    coefficient = float(
        result.params["machine_predictor"]
    )
    standard_error = float(
        result.bse["machine_predictor"]
    )
    p_value = float(
        result.pvalues["machine_predictor"]
    )

    return {
        "estimate": coefficient,
        "standard_error": standard_error,
        "ci95_low": (
            coefficient
            - 1.96 * standard_error
        ),
        "ci95_high": (
            coefficient
            + 1.96 * standard_error
        ),
        "odds_ratio": float(
            np.exp(coefficient)
        ),
        "odds_ratio_ci95_low": float(
            np.exp(
                coefficient
                - 1.96 * standard_error
            )
        ),
        "odds_ratio_ci95_high": float(
            np.exp(
                coefficient
                + 1.96 * standard_error
            )
        ),
        "p_value": p_value,
    }


primary_rows = []

for pair_row in CONSTRUCT_MAP.itertuples(
    index=False
):
    gate = model_support.loc[
        model_support["pair_id"].eq(
            pair_row.pair_id
        )
    ].iloc[0]

    data = prepare_pair_dataset(
        pair_row
    )

    if not bool(gate["support_pass"]):
        primary_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "inference_tier": (
                    pair_row.inference_tier
                ),
                "human_label": (
                    pair_row.human_label
                ),
                "feature": pair_row.feature,
                "feature_label": (
                    pair_row.feature_label
                ),
                "expected_sign": (
                    pair_row.expected_sign
                ),
                "predictor_type": (
                    pair_row.predictor_type
                ),
                "recordings": len(data),
                "participants": data[
                    "participant_id"
                ].nunique(),
                "human_positives": int(
                    data["human_present"].sum()
                ),
                "model_status": (
                    "support_gate_failed"
                ),
                "model_error": (
                    gate["support_reason"]
                ),
            }
        )
        continue

    result, error = fit_gee_model(
        data,
        correlation="exchangeable",
    )

    if result is None:
        primary_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "inference_tier": (
                    pair_row.inference_tier
                ),
                "human_label": (
                    pair_row.human_label
                ),
                "feature": pair_row.feature,
                "feature_label": (
                    pair_row.feature_label
                ),
                "expected_sign": (
                    pair_row.expected_sign
                ),
                "predictor_type": (
                    pair_row.predictor_type
                ),
                "recordings": len(data),
                "participants": data[
                    "participant_id"
                ].nunique(),
                "human_positives": int(
                    data["human_present"].sum()
                ),
                "model_status": "failed",
                "model_error": error,
            }
        )
        continue

    effect = extract_machine_effect(
        result
    )

    expected_sign = int(
        pair_row.expected_sign
    )

    oriented_estimate = (
        expected_sign
        * effect["estimate"]
        if expected_sign in {-1, 1}
        else np.nan
    )

    primary_rows.append(
        {
            "pair_id": pair_row.pair_id,
            "family": pair_row.family,
            "inference_tier": (
                pair_row.inference_tier
            ),
            "human_label": (
                pair_row.human_label
            ),
            "feature": pair_row.feature,
            "feature_label": (
                pair_row.feature_label
            ),
            "relationship_role": (
                pair_row.relationship_role
            ),
            "expected_sign": (
                pair_row.expected_sign
            ),
            "predictor_type": (
                pair_row.predictor_type
            ),
            "effect_scale": (
                data.attrs["effect_scale"]
            ),
            "recordings": len(data),
            "participants": data[
                "participant_id"
            ].nunique(),
            "human_positives": int(
                data["human_present"].sum()
            ),
            "human_negatives": int(
                (1 - data["human_present"]).sum()
            ),
            "working_correlation": float(
                np.asarray(
                    result.cov_struct.dep_params
                ).reshape(-1)[0]
            )
            if np.asarray(
                result.cov_struct.dep_params
            ).size
            else np.nan,
            "model_status": "fit",
            "model_error": "",
            **effect,
            "direction_oriented_estimate": (
                oriented_estimate
            ),
            "direction_oriented_odds_ratio": (
                float(
                    np.exp(
                        oriented_estimate
                    )
                )
                if np.isfinite(
                    oriented_estimate
                )
                else np.nan
            ),
            "estimate_matches_expected_direction": (
                bool(
                    oriented_estimate > 0
                )
                if np.isfinite(
                    oriented_estimate
                )
                else np.nan
            ),
        }
    )

primary_models = pd.DataFrame(
    primary_rows
)

primary_models = bh_adjust(
    primary_models,
    group_column="inference_tier",
)

display(
    primary_models[
        [
            "pair_id",
            "family",
            "inference_tier",
            "human_label",
            "feature_label",
            "recordings",
            "human_positives",
            "odds_ratio",
            "odds_ratio_ci95_low",
            "odds_ratio_ci95_high",
            "p_value",
            "q_value",
            "estimate_matches_expected_direction",
            "model_status",
        ]
    ]
)

export_dataframe(
    primary_models,
    TABLE_DIR / "primary_construct_matched_models.csv",
)


## 11. Independence-working-correlation sensitivity

In [ ]:
independence_rows = []

for pair_row in CONSTRUCT_MAP.itertuples(
    index=False
):
    gate = model_support.loc[
        model_support["pair_id"].eq(
            pair_row.pair_id
        )
    ].iloc[0]

    if not bool(gate["support_pass"]):
        continue

    data = prepare_pair_dataset(
        pair_row
    )

    result, error = fit_gee_model(
        data,
        correlation="independence",
    )

    if result is None:
        independence_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "model_status": "failed",
                "model_error": error,
            }
        )
        continue

    effect = extract_machine_effect(
        result
    )

    independence_rows.append(
        {
            "pair_id": pair_row.pair_id,
            "family": pair_row.family,
            "human_label": (
                pair_row.human_label
            ),
            "feature_label": (
                pair_row.feature_label
            ),
            "model_status": "fit",
            "model_error": "",
            **effect,
        }
    )

independence_sensitivity = pd.DataFrame(
    independence_rows
)

export_dataframe(
    independence_sensitivity,
    TABLE_DIR
    / "independence_working_correlation_sensitivity.csv",
)


## 12. Leave-one-rater-out sensitivity

In [ ]:
leave_one_rater_rows = []

for pair_row in CONSTRUCT_MAP.itertuples(
    index=False
):
    full_gate = model_support.loc[
        model_support["pair_id"].eq(
            pair_row.pair_id
        )
    ].iloc[0]

    if not bool(full_gate["support_pass"]):
        continue

    for omitted_rater in EXPECTED_RATERS:
        retained_raters = [
            rater
            for rater in EXPECTED_RATERS
            if rater != omitted_rater
        ]

        data = prepare_pair_dataset(
            pair_row,
            allowed_raters=retained_raters,
        )

        positives = data.loc[
            data["human_present"].eq(1)
        ]
        negatives = data.loc[
            data["human_present"].eq(0)
        ]

        support_pass = bool(
            len(data) >= MIN_MODEL_RECORDINGS
            and len(positives) >= 10
            and len(negatives) >= 10
            and positives[
                "participant_id"
            ].nunique()
            >= 5
            and negatives[
                "participant_id"
            ].nunique()
            >= 5
            and data[
                "machine_predictor"
            ].nunique()
            >= MIN_PREDICTOR_UNIQUE
        )

        if not support_pass:
            leave_one_rater_rows.append(
                {
                    "pair_id": pair_row.pair_id,
                    "family": pair_row.family,
                    "human_label": (
                        pair_row.human_label
                    ),
                    "feature_label": (
                        pair_row.feature_label
                    ),
                    "omitted_rater": (
                        omitted_rater
                    ),
                    "recordings": len(data),
                    "human_positives": int(
                        data[
                            "human_present"
                        ].sum()
                    ),
                    "model_status": (
                        "support_gate_failed"
                    ),
                }
            )
            continue

        result, error = fit_gee_model(
            data,
            correlation="exchangeable",
        )

        if result is None:
            leave_one_rater_rows.append(
                {
                    "pair_id": pair_row.pair_id,
                    "family": pair_row.family,
                    "human_label": (
                        pair_row.human_label
                    ),
                    "feature_label": (
                        pair_row.feature_label
                    ),
                    "omitted_rater": (
                        omitted_rater
                    ),
                    "recordings": len(data),
                    "human_positives": int(
                        data[
                            "human_present"
                        ].sum()
                    ),
                    "model_status": "failed",
                    "model_error": error,
                }
            )
            continue

        effect = extract_machine_effect(
            result
        )

        expected_sign = int(
            pair_row.expected_sign
        )
        oriented_estimate = (
            expected_sign
            * effect["estimate"]
            if expected_sign in {-1, 1}
            else np.nan
        )

        leave_one_rater_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "human_label": (
                    pair_row.human_label
                ),
                "feature_label": (
                    pair_row.feature_label
                ),
                "omitted_rater": (
                    omitted_rater
                ),
                "recordings": len(data),
                "participants": data[
                    "participant_id"
                ].nunique(),
                "human_positives": int(
                    data["human_present"].sum()
                ),
                "model_status": "fit",
                "model_error": "",
                **effect,
                "direction_oriented_estimate": (
                    oriented_estimate
                ),
                "estimate_matches_expected_direction": (
                    bool(
                        oriented_estimate > 0
                    )
                    if np.isfinite(
                        oriented_estimate
                    )
                    else np.nan
                ),
            }
        )

leave_one_rater = pd.DataFrame(
    leave_one_rater_rows
)

leave_one_rater_summary_rows = []

for pair_id, group in leave_one_rater.groupby(
    "pair_id",
    sort=False,
):
    fit_group = group.loc[
        group["model_status"].eq("fit")
    ]

    leave_one_rater_summary_rows.append(
        {
            "pair_id": pair_id,
            "successful_models": len(
                fit_group
            ),
            "minimum_odds_ratio": (
                float(
                    fit_group[
                        "odds_ratio"
                    ].min()
                )
                if len(fit_group)
                else np.nan
            ),
            "maximum_odds_ratio": (
                float(
                    fit_group[
                        "odds_ratio"
                    ].max()
                )
                if len(fit_group)
                else np.nan
            ),
            "expected_direction_models": (
                int(
                    fit_group[
                        "estimate_matches_expected_direction"
                    ].sum()
                )
                if len(fit_group)
                else 0
            ),
            "all_successful_models_match_expected_direction": (
                bool(
                    fit_group[
                        "estimate_matches_expected_direction"
                    ].all()
                )
                if len(fit_group)
                else False
            ),
        }
    )

leave_one_rater_summary = pd.DataFrame(
    leave_one_rater_summary_rows
)

export_dataframe(
    leave_one_rater,
    TABLE_DIR / "leave_one_rater_out_models.csv",
)
export_dataframe(
    leave_one_rater_summary,
    TABLE_DIR / "leave_one_rater_out_summary.csv",
)

display(leave_one_rater_summary)


## 13. Participant-collapsed resampling sensitivity

For each construct pair, one recording is sampled per participant on every replicate. A standard adjusted logistic model is then fitted. This evaluates whether the primary result is driven by unequal recording counts.


In [ ]:
def fit_subject_collapsed_glm(
    data: pd.DataFrame,
) -> float:
    sampled = (
        data.groupby(
            "participant_id",
            group_keys=False,
            observed=True,
        )
        .sample(
            n=1,
            replace=False,
            random_state=None,
        )
        .copy()
    )

    sampled["rater"] = (
        sampled["rater"]
        .astype("category")
        .cat.remove_unused_categories()
    )
    sampled["cohort"] = (
        sampled["cohort"]
        .astype("category")
        .cat.remove_unused_categories()
    )

    try:
        model = smf.glm(
            formula=(
                "human_present ~ machine_predictor "
                "+ C(rater) + C(cohort) "
                "+ log_recording_duration_centered"
            ),
            data=sampled,
            family=sm.families.Binomial(),
        )

        result = model.fit(
            maxiter=200,
            disp=0,
        )

        coefficient = float(
            result.params[
                "machine_predictor"
            ]
        )

        return coefficient

    except Exception:
        return np.nan


rng = np.random.default_rng(SEED)
subject_collapsed_rows = []

for pair_row in CONSTRUCT_MAP.itertuples(
    index=False
):
    gate = model_support.loc[
        model_support["pair_id"].eq(
            pair_row.pair_id
        )
    ].iloc[0]

    if not bool(gate["support_pass"]):
        continue

    data = prepare_pair_dataset(
        pair_row
    )

    coefficients = np.full(
        N_SUBJECT_COLLAPSED_RESAMPLES,
        np.nan,
        dtype=float,
    )

    # Draw participant-specific rows explicitly so the global seed is
    # deterministic across operating systems and pandas versions.
    participant_groups = {
        participant_id: group.index.to_numpy()
        for participant_id, group in data.groupby(
            "participant_id",
            observed=True,
        )
    }

    for replicate in range(
        N_SUBJECT_COLLAPSED_RESAMPLES
    ):
        selected_indices = [
            rng.choice(indices)
            for indices in participant_groups.values()
        ]

        sampled = data.loc[
            selected_indices
        ].copy()

        sampled["rater"] = (
            sampled["rater"]
            .astype("category")
            .cat.remove_unused_categories()
        )
        sampled["cohort"] = (
            sampled["cohort"]
            .astype("category")
            .cat.remove_unused_categories()
        )

        try:
            model = smf.glm(
                formula=(
                    "human_present ~ machine_predictor "
                    "+ C(rater) + C(cohort) "
                    "+ log_recording_duration_centered"
                ),
                data=sampled,
                family=sm.families.Binomial(),
            )

            result = model.fit(
                maxiter=200,
                disp=0,
            )

            coefficients[replicate] = float(
                result.params[
                    "machine_predictor"
                ]
            )

        except Exception:
            continue

    finite = coefficients[
        np.isfinite(coefficients)
    ]

    success_fraction = (
        len(finite)
        / N_SUBJECT_COLLAPSED_RESAMPLES
    )

    expected_sign = int(
        pair_row.expected_sign
    )

    oriented = (
        expected_sign * finite
        if expected_sign in {-1, 1}
        else np.full(
            len(finite),
            np.nan,
        )
    )

    subject_collapsed_rows.append(
        {
            "pair_id": pair_row.pair_id,
            "family": pair_row.family,
            "human_label": (
                pair_row.human_label
            ),
            "feature_label": (
                pair_row.feature_label
            ),
            "requested_resamples": (
                N_SUBJECT_COLLAPSED_RESAMPLES
            ),
            "successful_resamples": (
                len(finite)
            ),
            "success_fraction": (
                success_fraction
            ),
            "median_estimate": (
                float(
                    np.median(finite)
                )
                if len(finite)
                else np.nan
            ),
            "estimate_p025": (
                float(
                    np.quantile(
                        finite,
                        0.025,
                    )
                )
                if len(finite)
                else np.nan
            ),
            "estimate_p975": (
                float(
                    np.quantile(
                        finite,
                        0.975,
                    )
                )
                if len(finite)
                else np.nan
            ),
            "median_odds_ratio": (
                float(
                    np.exp(
                        np.median(finite)
                    )
                )
                if len(finite)
                else np.nan
            ),
            "expected_direction_fraction": (
                float(
                    np.mean(
                        oriented > 0
                    )
                )
                if np.isfinite(
                    oriented
                ).any()
                else np.nan
            ),
            "passes_success_gate": (
                success_fraction
                >= MIN_RESAMPLE_SUCCESS
            ),
        }
    )

subject_collapsed_sensitivity = pd.DataFrame(
    subject_collapsed_rows
)

display(subject_collapsed_sensitivity)

export_dataframe(
    subject_collapsed_sensitivity,
    TABLE_DIR
    / "participant_collapsed_resampling_sensitivity.csv",
)


## 14. Feature-by-rater slope heterogeneity

In [ ]:
interaction_rows = []

for pair_row in CONSTRUCT_MAP.itertuples(
    index=False
):
    gate = model_support.loc[
        model_support["pair_id"].eq(
            pair_row.pair_id
        )
    ].iloc[0]

    if not bool(gate["support_pass"]):
        continue

    data = prepare_pair_dataset(
        pair_row
    )

    try:
        model = smf.gee(
            formula=(
                "human_present ~ machine_predictor * C(rater) "
                "+ C(cohort) "
                "+ log_recording_duration_centered"
            ),
            groups="participant_id",
            data=data,
            family=sm.families.Binomial(),
            cov_struct=Exchangeable(),
        )

        result = model.fit(
            maxiter=250
        )

        interaction_terms = [
            term
            for term in result.params.index
            if (
                "machine_predictor:C(rater)"
                in term
                or "C(rater)" in term
                and ":machine_predictor"
                in term
            )
        ]

        if not interaction_terms:
            statistic = np.nan
            degrees_freedom = 0
            p_value = np.nan
        else:
            restriction = np.zeros(
                (
                    len(interaction_terms),
                    len(result.params),
                )
            )

            for row_index, term in enumerate(
                interaction_terms
            ):
                restriction[
                    row_index,
                    list(
                        result.params.index
                    ).index(term),
                ] = 1.0

            test = result.wald_test(
                restriction,
                scalar=True,
            )

            statistic = float(
                np.asarray(
                    test.statistic
                ).reshape(-1)[0]
            )
            degrees_freedom = len(
                interaction_terms
            )
            p_value = float(
                test.pvalue
            )

        interaction_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "human_label": (
                    pair_row.human_label
                ),
                "feature_label": (
                    pair_row.feature_label
                ),
                "interaction_terms": (
                    len(interaction_terms)
                ),
                "wald_statistic": statistic,
                "degrees_freedom": (
                    degrees_freedom
                ),
                "p_value": p_value,
                "model_status": "fit",
                "model_error": "",
            }
        )

    except Exception as exc:
        interaction_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "human_label": (
                    pair_row.human_label
                ),
                "feature_label": (
                    pair_row.feature_label
                ),
                "model_status": "failed",
                "model_error": (
                    f"{type(exc).__name__}: {exc}"
                ),
            }
        )

rater_slope_heterogeneity = pd.DataFrame(
    interaction_rows
)

if not rater_slope_heterogeneity.empty:
    valid = (
        rater_slope_heterogeneity[
            "p_value"
        ].notna()
    )

    rater_slope_heterogeneity[
        "q_value"
    ] = np.nan

    if valid.any():
        rater_slope_heterogeneity.loc[
            valid,
            "q_value",
        ] = multipletests(
            rater_slope_heterogeneity.loc[
                valid,
                "p_value",
            ],
            method="fdr_bh",
        )[1]

export_dataframe(
    rater_slope_heterogeneity,
    TABLE_DIR / "feature_by_rater_slope_heterogeneity.csv",
)


In [ ]:
# ---------------------------------------------------------------------
# Supplementary diagnostic:
# feature-by-rater slope heterogeneity
#
# These models are not part of the primary scientific inference.
# They are fitted only when every rater has sufficient positive and
# negative outcome support. Sparse or separated models are retained
# as non-estimable rather than interpreted.
# ---------------------------------------------------------------------

import warnings

MIN_RATER_POSITIVES_FOR_INTERACTION = 10
MIN_RATER_NEGATIVES_FOR_INTERACTION = 10
MIN_RATER_POSITIVE_PARTICIPANTS = 5
MIN_RATER_NEGATIVE_PARTICIPANTS = 5

interaction_rows = []
interaction_support_rows = []

for pair_row in CONSTRUCT_MAP.itertuples(index=False):

    gate = model_support.loc[
        model_support["pair_id"].eq(pair_row.pair_id)
    ].iloc[0]

    if not bool(gate["support_pass"]):
        interaction_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "human_label": pair_row.human_label,
                "feature_label": pair_row.feature_label,
                "interaction_terms": 0,
                "wald_statistic": np.nan,
                "degrees_freedom": 0,
                "p_value": np.nan,
                "model_converged": False,
                "maximum_absolute_parameter": np.nan,
                "numerical_warnings": "",
                "model_status": "primary_support_gate_failed",
                "model_error": str(gate["support_reason"]),
            }
        )
        continue

    data = prepare_pair_dataset(pair_row).copy()

    # -------------------------------------------------------------
    # Verify support separately within every rater.
    # -------------------------------------------------------------
    rater_support_records = []

    for rater in EXPECTED_RATERS:
        rater_data = data.loc[
            data["rater"].astype(str).eq(rater)
        ].copy()

        positive_data = rater_data.loc[
            rater_data["human_present"].eq(1)
        ]

        negative_data = rater_data.loc[
            rater_data["human_present"].eq(0)
        ]

        rater_support_records.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "human_label": pair_row.human_label,
                "feature_label": pair_row.feature_label,
                "rater": rater,
                "recordings": len(rater_data),
                "positives": len(positive_data),
                "negatives": len(negative_data),
                "positive_participants": (
                    positive_data["participant_id"].nunique()
                ),
                "negative_participants": (
                    negative_data["participant_id"].nunique()
                ),
            }
        )

    pair_rater_support = pd.DataFrame(rater_support_records)
    interaction_support_rows.extend(rater_support_records)

    sparse_raters = pair_rater_support.loc[
        (
            pair_rater_support["positives"]
            < MIN_RATER_POSITIVES_FOR_INTERACTION
        )
        | (
            pair_rater_support["negatives"]
            < MIN_RATER_NEGATIVES_FOR_INTERACTION
        )
        | (
            pair_rater_support["positive_participants"]
            < MIN_RATER_POSITIVE_PARTICIPANTS
        )
        | (
            pair_rater_support["negative_participants"]
            < MIN_RATER_NEGATIVE_PARTICIPANTS
        )
    ].copy()

    if not sparse_raters.empty:
        sparse_detail = " | ".join(
            (
                sparse_raters["rater"].astype(str)
                + ": pos="
                + sparse_raters["positives"].astype(str)
                + ", neg="
                + sparse_raters["negatives"].astype(str)
                + ", pos_participants="
                + sparse_raters["positive_participants"].astype(str)
                + ", neg_participants="
                + sparse_raters["negative_participants"].astype(str)
            ).tolist()
        )

        interaction_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "human_label": pair_row.human_label,
                "feature_label": pair_row.feature_label,
                "interaction_terms": 0,
                "wald_statistic": np.nan,
                "degrees_freedom": 0,
                "p_value": np.nan,
                "model_converged": False,
                "maximum_absolute_parameter": np.nan,
                "numerical_warnings": "",
                "model_status": "not_estimable_sparse_rater_cells",
                "model_error": sparse_detail,
            }
        )
        continue

    # Remove unused category levels before fitting.
    data["rater"] = (
        data["rater"]
        .astype("category")
        .cat.remove_unused_categories()
    )

    data["cohort"] = (
        data["cohort"]
        .astype("category")
        .cat.remove_unused_categories()
    )

    try:
        model = smf.gee(
            formula=(
                "human_present ~ machine_predictor * C(rater) "
                "+ C(cohort) "
                "+ log_recording_duration_centered"
            ),
            groups="participant_id",
            data=data,
            family=sm.families.Binomial(),
            cov_struct=Exchangeable(),
        )

        # Capture numerical warnings instead of printing them as if they
        # were notebook errors.
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", RuntimeWarning)

            result = model.fit(
                maxiter=250
            )

        warning_messages = sorted(
            {
                str(warning.message)
                for warning in caught_warnings
                if issubclass(
                    warning.category,
                    RuntimeWarning,
                )
            }
        )

        numerical_warning_text = " | ".join(warning_messages)

        model_converged = bool(
            getattr(
                result,
                "converged",
                True,
            )
        )

        parameters = pd.to_numeric(
            result.params,
            errors="coerce",
        )

        standard_errors = pd.to_numeric(
            result.bse,
            errors="coerce",
        )

        finite_parameters = bool(
            np.isfinite(parameters).all()
        )

        finite_standard_errors = bool(
            np.isfinite(standard_errors).all()
        )

        maximum_absolute_parameter = (
            float(np.max(np.abs(parameters)))
            if finite_parameters and len(parameters)
            else np.nan
        )

        interaction_terms = [
            term
            for term in result.params.index
            if (
                "machine_predictor:C(rater)" in term
                or (
                    "C(rater)" in term
                    and ":machine_predictor" in term
                )
            )
        ]

        has_overflow_warning = any(
            (
                "overflow" in message.casefold()
                or "invalid value" in message.casefold()
            )
            for message in warning_messages
        )

        numerically_stable = bool(
            model_converged
            and finite_parameters
            and finite_standard_errors
            and not has_overflow_warning
        )

        if not interaction_terms:
            interaction_rows.append(
                {
                    "pair_id": pair_row.pair_id,
                    "family": pair_row.family,
                    "human_label": pair_row.human_label,
                    "feature_label": pair_row.feature_label,
                    "interaction_terms": 0,
                    "wald_statistic": np.nan,
                    "degrees_freedom": 0,
                    "p_value": np.nan,
                    "model_converged": model_converged,
                    "maximum_absolute_parameter": (
                        maximum_absolute_parameter
                    ),
                    "numerical_warnings": numerical_warning_text,
                    "model_status": "interaction_terms_not_identified",
                    "model_error": (
                        "No machine-predictor-by-rater terms were "
                        "identified in the fitted model."
                    ),
                }
            )
            continue

        if not numerically_stable:
            interaction_rows.append(
                {
                    "pair_id": pair_row.pair_id,
                    "family": pair_row.family,
                    "human_label": pair_row.human_label,
                    "feature_label": pair_row.feature_label,
                    "interaction_terms": len(interaction_terms),
                    "wald_statistic": np.nan,
                    "degrees_freedom": len(interaction_terms),
                    "p_value": np.nan,
                    "model_converged": model_converged,
                    "maximum_absolute_parameter": (
                        maximum_absolute_parameter
                    ),
                    "numerical_warnings": numerical_warning_text,
                    "model_status": "unstable_numerical_fit",
                    "model_error": (
                        "The interaction model showed numerical "
                        "overflow, non-finite estimates, or failed "
                        "convergence and was not interpreted."
                    ),
                }
            )
            continue

        restriction = np.zeros(
            (
                len(interaction_terms),
                len(result.params),
            ),
            dtype=float,
        )

        parameter_names = list(
            result.params.index
        )

        for row_index, term in enumerate(
            interaction_terms
        ):
            restriction[
                row_index,
                parameter_names.index(term),
            ] = 1.0

        test = result.wald_test(
            restriction,
            scalar=True,
        )

        statistic = float(
            np.asarray(
                test.statistic
            ).reshape(-1)[0]
        )

        degrees_freedom = len(
            interaction_terms
        )

        p_value = float(
            test.pvalue
        )

        interaction_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "human_label": pair_row.human_label,
                "feature_label": pair_row.feature_label,
                "interaction_terms": len(interaction_terms),
                "wald_statistic": statistic,
                "degrees_freedom": degrees_freedom,
                "p_value": p_value,
                "model_converged": model_converged,
                "maximum_absolute_parameter": (
                    maximum_absolute_parameter
                ),
                "numerical_warnings": numerical_warning_text,
                "model_status": "fit",
                "model_error": "",
            }
        )

    except Exception as exc:
        interaction_rows.append(
            {
                "pair_id": pair_row.pair_id,
                "family": pair_row.family,
                "human_label": pair_row.human_label,
                "feature_label": pair_row.feature_label,
                "interaction_terms": 0,
                "wald_statistic": np.nan,
                "degrees_freedom": 0,
                "p_value": np.nan,
                "model_converged": False,
                "maximum_absolute_parameter": np.nan,
                "numerical_warnings": "",
                "model_status": "failed",
                "model_error": (
                    f"{type(exc).__name__}: {exc}"
                ),
            }
        )


interaction_columns = [
    "pair_id",
    "family",
    "human_label",
    "feature_label",
    "interaction_terms",
    "wald_statistic",
    "degrees_freedom",
    "p_value",
    "model_converged",
    "maximum_absolute_parameter",
    "numerical_warnings",
    "model_status",
    "model_error",
]

rater_slope_heterogeneity = pd.DataFrame(
    interaction_rows,
    columns=interaction_columns,
)

rater_interaction_support = pd.DataFrame(
    interaction_support_rows
)

rater_slope_heterogeneity["q_value"] = np.nan

valid = (
    rater_slope_heterogeneity["model_status"].eq("fit")
    & rater_slope_heterogeneity["p_value"].notna()
    & np.isfinite(
        rater_slope_heterogeneity["p_value"]
    )
)

if valid.any():
    rater_slope_heterogeneity.loc[
        valid,
        "q_value",
    ] = multipletests(
        rater_slope_heterogeneity.loc[
            valid,
            "p_value",
        ].astype(float),
        method="fdr_bh",
    )[1]


display(
    rater_slope_heterogeneity[
        [
            "pair_id",
            "family",
            "human_label",
            "feature_label",
            "model_status",
            "p_value",
            "q_value",
            "model_error",
        ]
    ]
)

export_dataframe(
    rater_slope_heterogeneity,
    TABLE_DIR
    / "feature_by_rater_slope_heterogeneity.csv",
)

export_dataframe(
    rater_interaction_support,
    TABLE_DIR
    / "feature_by_rater_interaction_support.csv",
)

## 15. QREV conditional-support and QDIST/QTEMP sparse descriptives

QREV boundary-tail models are not interpreted inferentially when the available subset contains fewer than 20 human-positive recordings. QDIST and QTEMP are summarized by machine-positive and human-positive cross-tabulation only.


In [ ]:
conditional_descriptive_rows = []

for pair_id in [
    "FAM_QREV_02",
    "FAM_QREV_03",
]:
    pair_row = CONSTRUCT_MAP.loc[
        CONSTRUCT_MAP["pair_id"].eq(
            pair_id
        )
    ].iloc[0]

    data = prepare_pair_dataset(
        pair_row
    )

    for human_state, group in data.groupby(
        "human_present"
    ):
        conditional_descriptive_rows.append(
            {
                "pair_id": pair_id,
                "family": "QREV",
                "human_label": (
                    pair_row["human_label"]
                ),
                "feature": (
                    pair_row["feature"]
                ),
                "feature_label": (
                    pair_row["feature_label"]
                ),
                "human_present": int(
                    human_state
                ),
                "recordings": len(group),
                "participants": group[
                    "participant_id"
                ].nunique(),
                "median_machine_predictor": float(
                    group[
                        "machine_predictor"
                    ].median()
                ),
                "iqr_machine_predictor": float(
                    group[
                        "machine_predictor"
                    ].quantile(0.75)
                    - group[
                        "machine_predictor"
                    ].quantile(0.25)
                ),
            }
        )

conditional_qrev_descriptives = pd.DataFrame(
    conditional_descriptive_rows
)


def sparse_cross_tab(
    human_level: str,
    human_value: str,
    human_label: str,
    feature: str,
    feature_label: str,
    family: str,
) -> dict[str, Any]:
    human = extract_human_outcome(
        human_level,
        human_value,
    )

    data = analysis_519[
        [
            "recording_id",
            "participant_id",
            "cohort",
            feature,
            f"{feature}__positive",
        ]
    ].merge(
        human,
        on="recording_id",
        how="inner",
        validate="one_to_one",
    )

    machine_positive = data[
        f"{feature}__positive"
    ].astype(bool)
    human_positive = data[
        "human_present"
    ].astype(bool)

    n11 = int(
        (machine_positive & human_positive).sum()
    )
    n10 = int(
        (machine_positive & ~human_positive).sum()
    )
    n01 = int(
        (~machine_positive & human_positive).sum()
    )
    n00 = int(
        (~machine_positive & ~human_positive).sum()
    )

    odds_ratio, fisher_p = stats.fisher_exact(
        [
            [n11, n10],
            [n01, n00],
        ],
        alternative="two-sided",
    )

    return {
        "family": family,
        "human_label": human_label,
        "feature": feature,
        "feature_label": feature_label,
        "recordings": len(data),
        "machine_positive_recordings": int(
            machine_positive.sum()
        ),
        "human_positive_recordings": int(
            human_positive.sum()
        ),
        "both_positive_n11": n11,
        "machine_only_n10": n10,
        "human_only_n01": n01,
        "both_negative_n00": n00,
        "fisher_odds_ratio_descriptive": (
            float(odds_ratio)
            if np.isfinite(odds_ratio)
            else np.inf
        ),
        "fisher_p_descriptive": float(
            fisher_p
        ),
        "inferential_role": "descriptive only",
    }


sparse_rows = [
    sparse_cross_tab(
        "signal_family",
        "QDIST",
        "Human clipping/overload",
        "qdist_hard_clipped_sample_fraction",
        "Machine hard-clipping occurrence",
        "QDIST",
    ),
    sparse_cross_tab(
        "signal_family",
        "QTEMP",
        "Human temporal discontinuity",
        "qtemp_dropout_duration_fraction",
        "Machine dropout-like occurrence",
        "QTEMP",
    ),
    sparse_cross_tab(
        "signal_family",
        "QTEMP",
        "Human temporal discontinuity",
        "qtemp_frozen_audio_duration_fraction",
        "Machine frozen-audio occurrence",
        "QTEMP",
    ),
]

sparse_correspondence = pd.DataFrame(
    sparse_rows
)

display(conditional_qrev_descriptives)
display(sparse_correspondence)

export_dataframe(
    conditional_qrev_descriptives,
    TABLE_DIR / "qrev_conditional_support_descriptives.csv",
)
export_dataframe(
    sparse_correspondence,
    TABLE_DIR / "qdist_qtemp_sparse_correspondence.csv",
)


## 16. Integrated decision table

In [ ]:
integrated_decision = (
    CONSTRUCT_MAP.merge(
        model_support[
            [
                "pair_id",
                "recordings",
                "participants",
                "human_positives",
                "human_negatives",
                "support_pass",
                "support_reason",
            ]
        ],
        on="pair_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        primary_models[
            [
                "pair_id",
                "estimate",
                "standard_error",
                "ci95_low",
                "ci95_high",
                "odds_ratio",
                "odds_ratio_ci95_low",
                "odds_ratio_ci95_high",
                "p_value",
                "q_value",
                "estimate_matches_expected_direction",
                "model_status",
            ]
        ],
        on="pair_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        leave_one_rater_summary,
        on="pair_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        subject_collapsed_sensitivity[
            [
                "pair_id",
                "success_fraction",
                "median_odds_ratio",
                "expected_direction_fraction",
                "passes_success_gate",
            ]
        ],
        on="pair_id",
        how="left",
        validate="one_to_one",
    )
)

def classify_correspondence(
    row: pd.Series,
) -> str:
    if not bool(row["support_pass"]):
        return "descriptive: insufficient joint support"

    if (
        row["model_status"] != "fit"
        or not np.isfinite(
            row["odds_ratio"]
        )
    ):
        return "model unavailable"

    q_significant = bool(
        np.isfinite(row["q_value"])
        and row["q_value"] < 0.05
    )

    expected = bool(
        row[
            "estimate_matches_expected_direction"
        ]
    )

    loo_stable = bool(
        row.get(
            "successful_models",
            0,
        )
        >= 3
        and row.get(
            "all_successful_models_match_expected_direction",
            False,
        )
    )

    collapsed_stable = bool(
        row.get(
            "passes_success_gate",
            False,
        )
        and np.isfinite(
            row.get(
                "expected_direction_fraction",
                np.nan,
            )
        )
        and row[
            "expected_direction_fraction"
        ]
        >= 0.80
    )

    if (
        row["inference_tier"]
        == "primary_family"
        and q_significant
        and expected
        and loo_stable
        and collapsed_stable
    ):
        return "supported family-level correspondence"

    if (
        row["inference_tier"]
        == "secondary_item"
        and q_significant
        and expected
        and loo_stable
    ):
        return "supported item-level refinement"

    if (
        q_significant
        and expected
        and loo_stable
    ):
        return "supported but sensitivity-limited correspondence"

    if q_significant and not expected:
        return "association opposite prespecified direction"

    if expected and loo_stable:
        return "directionally consistent but uncertain"

    return "no stable correspondence"


integrated_decision[
    "correspondence_class"
] = integrated_decision.apply(
    classify_correspondence,
    axis=1,
)

display(
    integrated_decision[
        [
            "pair_id",
            "family",
            "inference_tier",
            "human_label",
            "feature_label",
            "recordings",
            "human_positives",
            "odds_ratio",
            "odds_ratio_ci95_low",
            "odds_ratio_ci95_high",
            "q_value",
            "expected_direction_fraction",
            "correspondence_class",
        ]
    ]
)

export_dataframe(
    integrated_decision,
    TABLE_DIR / "integrated_correspondence_decisions.csv",
)


## 17. Cross-family specificity analysis

The main correspondence models test prespecified within-family matches. This analysis asks a different question: are artifact-oriented signal indicators more strongly associated with their intended human family than with unrelated human families?

Each core feature is modeled against QADD, QGAIN, QREV, and QCHAN family presence using the same participant-grouped, rater-, cohort-, and duration-adjusted GEE. An exact 4! family-label permutation compares the mean matched oriented Wald statistic with the mean mismatched statistic. This is an exploratory specificity test, not a redefinition of the family taxonomy.


In [ ]:
SPECIFICITY_HUMAN_FAMILIES = [
    "QADD",
    "QGAIN",
    "QREV",
    "QCHAN",
]


def prepare_specificity_dataset(
    feature_row: Any,
    human_family_value: str,
) -> pd.DataFrame:
    human = extract_human_outcome(
        "signal_family",
        human_family_value,
    )

    data = analysis_519.merge(
        human,
        on="recording_id",
        how="inner",
        validate="one_to_one",
    )

    if (
        feature_row.predictor_type
        == "continuous"
    ):
        data["machine_predictor"] = data[
            f"{feature_row.feature}__z"
        ]
    elif (
        feature_row.predictor_type
        == "binary_state"
    ):
        data["machine_predictor"] = data[
            f"{feature_row.feature}__positive"
        ].astype(float)
    else:
        raise ValueError(
            feature_row.predictor_type
        )

    data[
        "log_recording_duration_centered"
    ] = (
        np.log(
            data[
                "recording_duration_sec"
            ]
        )
        - LOG_DURATION_CENTER
    )

    data = data[
        [
            "recording_id",
            "participant_id",
            "cohort",
            "rater",
            "log_recording_duration_centered",
            "human_present",
            "machine_predictor",
        ]
    ].dropna()

    data["human_present"] = data[
        "human_present"
    ].astype(int)
    data["rater"] = data[
        "rater"
    ].astype("category")
    data["cohort"] = pd.Categorical(
        data["cohort"],
        categories=[
            "CONTROL",
            "ALS",
        ],
    )

    return data


specificity_rows = []

for feature_row in (
    SPECIFICITY_FEATURE_MAP.itertuples(
        index=False
    )
):
    for human_family_value in (
        SPECIFICITY_HUMAN_FAMILIES
    ):
        data = prepare_specificity_dataset(
            feature_row,
            human_family_value,
        )

        positives = int(
            data[
                "human_present"
            ].sum()
        )
        negatives = int(
            (
                1
                - data[
                    "human_present"
                ]
            ).sum()
        )

        predictor_support = (
            data[
                "machine_predictor"
            ].nunique()
            >= MIN_PREDICTOR_UNIQUE
        )

        if (
            feature_row.predictor_type
            == "binary_state"
        ):
            predictor_support = bool(
                predictor_support
                and data[
                    "machine_predictor"
                ].gt(0).sum()
                >= 10
                and data[
                    "machine_predictor"
                ].eq(0).sum()
                >= 10
            )

        support_pass = bool(
            len(data)
            >= MIN_MODEL_RECORDINGS
            and positives
            >= MIN_HUMAN_POSITIVES
            and negatives
            >= MIN_HUMAN_NEGATIVES
            and predictor_support
        )

        if not support_pass:
            specificity_rows.append(
                {
                    "feature_family": (
                        feature_row.family
                    ),
                    "human_family": (
                        human_family_value
                    ),
                    "feature": (
                        feature_row.feature
                    ),
                    "feature_label": (
                        feature_row.feature_label
                    ),
                    "expected_sign": (
                        feature_row.expected_sign
                    ),
                    "recordings": len(data),
                    "human_positives": (
                        positives
                    ),
                    "matched_family": (
                        feature_row.family
                        == human_family_value
                    ),
                    "model_status": (
                        "support_gate_failed"
                    ),
                }
            )
            continue

        result, error = fit_gee_model(
            data,
            correlation="exchangeable",
        )

        if result is None:
            specificity_rows.append(
                {
                    "feature_family": (
                        feature_row.family
                    ),
                    "human_family": (
                        human_family_value
                    ),
                    "feature": (
                        feature_row.feature
                    ),
                    "feature_label": (
                        feature_row.feature_label
                    ),
                    "expected_sign": (
                        feature_row.expected_sign
                    ),
                    "recordings": len(data),
                    "human_positives": (
                        positives
                    ),
                    "matched_family": (
                        feature_row.family
                        == human_family_value
                    ),
                    "model_status": "failed",
                    "model_error": error,
                }
            )
            continue

        effect = extract_machine_effect(
            result
        )

        oriented_beta = (
            feature_row.expected_sign
            * effect["estimate"]
        )
        oriented_z = (
            feature_row.expected_sign
            * effect["estimate"]
            / effect["standard_error"]
        )

        specificity_rows.append(
            {
                "feature_family": (
                    feature_row.family
                ),
                "human_family": (
                    human_family_value
                ),
                "feature": (
                    feature_row.feature
                ),
                "feature_label": (
                    feature_row.feature_label
                ),
                "expected_sign": (
                    feature_row.expected_sign
                ),
                "recordings": len(data),
                "human_positives": (
                    positives
                ),
                "matched_family": (
                    feature_row.family
                    == human_family_value
                ),
                "model_status": "fit",
                "model_error": "",
                **effect,
                "artifact_oriented_beta": (
                    oriented_beta
                ),
                "artifact_oriented_odds_ratio": float(
                    np.exp(
                        oriented_beta
                    )
                ),
                "artifact_oriented_z": (
                    oriented_z
                ),
            }
        )

specificity_models = pd.DataFrame(
    specificity_rows
)

specificity_fit = specificity_models.loc[
    specificity_models[
        "model_status"
    ].eq("fit")
].copy()

if specificity_fit.empty:
    raise RuntimeError(
        "No specificity models were estimable."
    )

observed_specificity_statistic = (
    specificity_fit.loc[
        specificity_fit[
            "matched_family"
        ],
        "artifact_oriented_z",
    ].mean()
    - specificity_fit.loc[
        ~specificity_fit[
            "matched_family"
        ],
        "artifact_oriented_z",
    ].mean()
)

permutation_rows = []

for permutation_index, permutation in enumerate(
    itertools.permutations(
        SPECIFICITY_HUMAN_FAMILIES
    ),
    start=1,
):
    assigned_family = dict(
        zip(
            SPECIFICITY_HUMAN_FAMILIES,
            permutation,
        )
    )

    selected = specificity_fit.apply(
        lambda row: (
            row["human_family"]
            == assigned_family[
                row["feature_family"]
            ]
        ),
        axis=1,
    )

    statistic = (
        specificity_fit.loc[
            selected,
            "artifact_oriented_z",
        ].mean()
        - specificity_fit.loc[
            ~selected,
            "artifact_oriented_z",
        ].mean()
    )

    permutation_rows.append(
        {
            "permutation_index": (
                permutation_index
            ),
            "assignment": " | ".join(
                f"{family}->{assigned_family[family]}"
                for family in (
                    SPECIFICITY_HUMAN_FAMILIES
                )
            ),
            "specificity_statistic": (
                statistic
            ),
            "is_observed_assignment": all(
                assigned_family[family]
                == family
                for family in (
                    SPECIFICITY_HUMAN_FAMILIES
                )
            ),
        }
    )

specificity_permutations = pd.DataFrame(
    permutation_rows
)

specificity_exact_p = float(
    (
        specificity_permutations[
            "specificity_statistic"
        ]
        >= observed_specificity_statistic
        - 1e-12
    ).mean()
)

specificity_summary = pd.DataFrame(
    [
        {
            "matched_mean_artifact_oriented_z": (
                specificity_fit.loc[
                    specificity_fit[
                        "matched_family"
                    ],
                    "artifact_oriented_z",
                ].mean()
            ),
            "mismatched_mean_artifact_oriented_z": (
                specificity_fit.loc[
                    ~specificity_fit[
                        "matched_family"
                    ],
                    "artifact_oriented_z",
                ].mean()
            ),
            "observed_specificity_statistic": (
                observed_specificity_statistic
            ),
            "exact_family_label_permutation_p": (
                specificity_exact_p
            ),
            "permutations": len(
                specificity_permutations
            ),
        }
    ]
)

display(
    specificity_summary
)

export_dataframe(
    specificity_models,
    TABLE_DIR
    / "cross_family_specificity_models.csv",
)
export_dataframe(
    specificity_permutations,
    TABLE_DIR
    / "cross_family_specificity_permutations.csv",
)
export_dataframe(
    specificity_summary,
    TABLE_DIR
    / "cross_family_specificity_summary.csv",
)


## 18. Main publication figure

In [ ]:
mpl.rcParams.update(
    {
        "font.family": [
            "Arial",
            "DejaVu Sans",
        ],
        "font.size": 7.2,
        "axes.titlesize": 8.6,
        "axes.labelsize": 7.4,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "legend.fontsize": 6.4,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)

family_order = [
    "QADD",
    "QGAIN",
    "QREV",
    "QCHAN",
    "QDIST",
    "QTEMP",
]

FAMILY_COLORS = {
    "QADD": "#1f77b4",
    "QGAIN": "#ff7f0e",
    "QREV": "#2ca02c",
    "QCHAN": "#d62728",
}

reliability_plot = (
    human_reliability_context.copy()
)

reliability_plot[
    "family_order"
] = reliability_plot[
    "signal_family"
].map(
    {
        family: index
        for index, family in enumerate(
            family_order
        )
    }
)

reliability_plot = (
    reliability_plot.sort_values(
        "family_order"
    )
)

forest_plot = integrated_decision.loc[
    integrated_decision[
        "inference_tier"
    ].eq("primary_family")
    & integrated_decision[
        "main_display"
    ]
    & integrated_decision[
        "support_pass"
    ]
    & integrated_decision[
        "model_status"
    ].eq("fit")
].copy()

forest_plot[
    "oriented_estimate"
] = (
    forest_plot[
        "expected_sign"
    ]
    * forest_plot[
        "estimate"
    ]
)

forest_plot[
    "oriented_or"
] = np.exp(
    forest_plot[
        "oriented_estimate"
    ]
)

forest_plot[
    "oriented_ci_low"
] = np.exp(
    np.minimum(
        forest_plot[
            "expected_sign"
        ]
        * forest_plot[
            "ci95_low"
        ],
        forest_plot[
            "expected_sign"
        ]
        * forest_plot[
            "ci95_high"
        ],
    )
)

forest_plot[
    "oriented_ci_high"
] = np.exp(
    np.maximum(
        forest_plot[
            "expected_sign"
        ]
        * forest_plot[
            "ci95_low"
        ],
        forest_plot[
            "expected_sign"
        ]
        * forest_plot[
            "ci95_high"
        ],
    )
)

forest_plot[
    "family_order"
] = forest_plot[
    "family"
].map(
    {
        family: index
        for index, family in enumerate(
            family_order
        )
    }
)

forest_plot = forest_plot.sort_values(
    [
        "family_order",
        "pair_id",
    ]
).reset_index(
    drop=True
)

forest_plot[
    "display_label"
] = (
    forest_plot[
        "family"
    ]
    + "  "
    + forest_plot[
        "feature_label"
    ]
)

specificity_feature_order = (
    SPECIFICITY_FEATURE_MAP.assign(
        family_order=lambda frame: (
            frame["family"].map(
                {
                    family: index
                    for index, family in enumerate(
                        [
                            "QADD",
                            "QGAIN",
                            "QREV",
                            "QCHAN",
                        ]
                    )
                }
            )
        ),
        map_order=np.arange(
            len(
                SPECIFICITY_FEATURE_MAP
            )
        ),
    )
    .sort_values(
        [
            "family_order",
            "map_order",
        ]
    )
)

specificity_row_order = (
    specificity_feature_order[
        "feature"
    ].tolist()
)

specificity_matrix = (
    specificity_fit.pivot(
        index="feature",
        columns="human_family",
        values="artifact_oriented_beta",
    )
    .reindex(
        index=specificity_row_order,
        columns=SPECIFICITY_HUMAN_FAMILIES,
    )
)

specificity_labels = (
    specificity_feature_order.set_index(
        "feature"
    )
    .loc[
        specificity_row_order,
        [
            "family",
            "feature_label",
        ],
    ]
)

export_dataframe(
    reliability_plot,
    FIGURE_SOURCE_DIR
    / "main_panel_A_human_reliability_context.csv",
)
export_dataframe(
    forest_plot,
    FIGURE_SOURCE_DIR
    / "main_panel_B_family_correspondence.csv",
)
export_dataframe(
    specificity_fit,
    FIGURE_SOURCE_DIR
    / "main_panel_C_specificity_models.csv",
)

figure = plt.figure(
    figsize=(
        7.35,
        9.35,
    ),
    facecolor="white",
)

grid = figure.add_gridspec(
    3,
    1,
    height_ratios=[
        1.15,
        2.65,
        2.75,
    ],
    hspace=0.58,
)

axis_reliability = figure.add_subplot(
    grid[0, 0]
)
axis_forest = figure.add_subplot(
    grid[1, 0]
)
axis_specificity = figure.add_subplot(
    grid[2, 0]
)

figure.suptitle(
    (
        "Family-dependent perceptual correspondence "
        "of recording-quality indicators"
    ),
    y=0.985,
    fontsize=10.6,
    fontweight="bold",
)

# --------------------------------------------------------------
# A. Reliability context — not the primary scientific endpoint.
# --------------------------------------------------------------
positions = np.arange(
    len(reliability_plot)
)

axis_reliability.scatter(
    positions - 0.09,
    reliability_plot[
        "gwet_ac1"
    ],
    marker="o",
    label="Gwet AC1",
)

axis_reliability.scatter(
    positions + 0.09,
    reliability_plot[
        "mean_pairwise_positive_agreement"
    ],
    marker="s",
    label="Positive agreement",
)

axis_reliability.set_xticks(
    positions
)
axis_reliability.set_xticklabels(
    reliability_plot[
        "signal_family"
    ]
)
axis_reliability.set_ylim(
    -0.02,
    1.02,
)
axis_reliability.set_ylabel(
    "Agreement"
)
axis_reliability.set_title(
    (
        "A  Reproducibility of the human "
        "family labels in the shared subset"
    ),
    loc="left",
    fontweight="bold",
)
axis_reliability.legend(
    frameon=False,
    ncol=2,
    loc="upper right",
)
axis_reliability.grid(
    axis="y",
    alpha=0.25,
)

# --------------------------------------------------------------
# B. Family-grouped construct-matched effects.
# --------------------------------------------------------------
forest_y = np.arange(
    len(forest_plot)
)

for row_index, row in forest_plot.iterrows():
    significant = bool(
        np.isfinite(
            row["q_value"]
        )
        and row["q_value"] < 0.05
    )

    family_color = FAMILY_COLORS[
        row["family"]
    ]

    axis_forest.errorbar(
        row["oriented_or"],
        row_index,
        xerr=[
            [
                row["oriented_or"]
                - row[
                    "oriented_ci_low"
                ]
            ],
            [
                row[
                    "oriented_ci_high"
                ]
                - row["oriented_or"]
            ],
        ],
        fmt="o",
        color=family_color,
        markeredgecolor=family_color,
        markerfacecolor=(
            family_color
            if significant
            else "white"
        ),
        capsize=2,
    )

axis_forest.axvline(
    1,
    linewidth=0.8,
)
axis_forest.set_xscale(
    "log"
)
axis_forest.set_xlim(
    0.20,
    4.6,
)
axis_forest.set_xticks(
    [
        0.25,
        0.5,
        1.0,
        2.0,
        4.0,
    ]
)
axis_forest.get_xaxis().set_major_formatter(
    mpl.ticker.ScalarFormatter()
)
axis_forest.set_yticks(
    forest_y
)
axis_forest.set_yticklabels(
    forest_plot[
        "display_label"
    ]
)
axis_forest.invert_yaxis()
axis_forest.set_xlabel(
    (
        "Direction-oriented odds ratio "
        "(95% CI; >1 follows the prespecified construct direction)"
    )
)
axis_forest.set_title(
    (
        "B  Signal indicators versus "
        "construct-matched human family presence"
    ),
    loc="left",
    fontweight="bold",
)
axis_forest.grid(
    axis="x",
    alpha=0.25,
)

# Add restrained family separators.
for row_index in range(
    1,
    len(forest_plot)
):
    if (
        forest_plot.loc[
            row_index,
            "family",
        ]
        != forest_plot.loc[
            row_index - 1,
            "family",
        ]
    ):
        axis_forest.axhline(
            row_index - 0.5,
            linewidth=0.5,
            alpha=0.35,
        )

# --------------------------------------------------------------
# C. Specificity matrix.
# --------------------------------------------------------------
finite_values = (
    specificity_matrix.to_numpy()
)
finite_values = finite_values[
    np.isfinite(
        finite_values
    )
]

color_limit = (
    float(
        np.quantile(
            np.abs(
                finite_values
            ),
            0.98,
        )
    )
    if finite_values.size
    else 1.0
)

color_limit = max(
    color_limit,
    0.25,
)

image = axis_specificity.imshow(
    specificity_matrix.to_numpy(),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-color_limit,
    vmax=color_limit,
)

axis_specificity.set_xticks(
    np.arange(
        len(
            SPECIFICITY_HUMAN_FAMILIES
        )
    )
)
axis_specificity.set_xticklabels(
    SPECIFICITY_HUMAN_FAMILIES
)

axis_specificity.set_yticks(
    np.arange(
        len(
            specificity_row_order
        )
    )
)
axis_specificity.set_yticklabels(
    [
        (
            specificity_labels.loc[
                feature,
                "family",
            ]
            + "  "
            + specificity_labels.loc[
                feature,
                "feature_label",
            ]
        )
        for feature in (
            specificity_row_order
        )
    ]
)

axis_specificity.set_title(
    (
        "C  Adjusted cross-family specificity; "
        "outlined cells are construct-matched"
    ),
    loc="left",
    fontweight="bold",
)

for row_index, feature in enumerate(
    specificity_row_order
):
    feature_family = (
        specificity_labels.loc[
            feature,
            "family",
        ]
    )

    if feature_family in (
        SPECIFICITY_HUMAN_FAMILIES
    ):
        column_index = (
            SPECIFICITY_HUMAN_FAMILIES.index(
                feature_family
            )
        )

        rectangle = mpl.patches.Rectangle(
            (
                column_index - 0.5,
                row_index - 0.5,
            ),
            1,
            1,
            fill=False,
            linewidth=1.25,
            edgecolor="black",
        )
        axis_specificity.add_patch(
            rectangle
        )

colorbar = figure.colorbar(
    image,
    ax=axis_specificity,
    fraction=0.026,
    pad=0.025,
)
colorbar.set_label(
    (
        "Artifact-oriented adjusted "
        "log-odds coefficient"
    )
)

axis_specificity.text(
    0.0,
    -0.18,
    (
        "Matched specificity contrast: "
        f"exact family-label permutation "
        f"p={specificity_exact_p:.3f}"
    ),
    transform=axis_specificity.transAxes,
    ha="left",
    va="top",
    fontsize=6.6,
)

figure.subplots_adjust(
    top=0.94,
    bottom=0.075,
)

main_paths = save_figure(
    figure,
    (
        "Figure_human_perceptual_"
        "correspondence_PUBLICATION"
    ),
    main=True,
)

plt.show()

print(main_paths)


## 19. Supplementary diagnostic figures

In [ ]:
# Supplementary figure: rater prevalence for all six families.
prevalence_columns = [
    f"prevalence_{rater}"
    for rater in EXPECTED_RATERS
]

prevalence_plot = reliability_family.loc[
    reliability_family[
        "signal_family"
    ].isin(family_order)
].copy()

prevalence_plot[
    "family_order"
] = prevalence_plot[
    "signal_family"
].map(
    {
        family: index
        for index, family in enumerate(
            family_order
        )
    }
)
prevalence_plot = prevalence_plot.sort_values(
    "family_order"
)

figure, axis = plt.subplots(
    figsize=(6.5, 3.7),
    constrained_layout=True,
)

for rater in EXPECTED_RATERS:
    axis.plot(
        prevalence_plot[
            "signal_family"
        ],
        100
        * prevalence_plot[
            f"prevalence_{rater}"
        ],
        marker="o",
        label=rater,
    )

axis.set_ylabel(
    "Positive ratings (%)"
)
axis.set_title(
    "Human family-label prevalence by rater",
    loc="left",
    fontweight="bold",
)
axis.legend(
    frameon=False,
    ncol=4,
)
axis.grid(
    axis="y",
    alpha=0.25,
)

save_figure(
    figure,
    "Supplementary_human_prevalence_by_rater",
    main=False,
)
plt.show()


# Supplementary figure: item-level refinements.
item_plot = integrated_decision.loc[
    integrated_decision[
        "inference_tier"
    ].eq("secondary_item")
    & integrated_decision[
        "support_pass"
    ]
    & integrated_decision[
        "model_status"
    ].eq("fit")
].copy()

item_plot[
    "oriented_estimate"
] = (
    item_plot[
        "expected_sign"
    ]
    * item_plot[
        "estimate"
    ]
)
item_plot[
    "oriented_or"
] = np.exp(
    item_plot[
        "oriented_estimate"
    ]
)
item_plot[
    "oriented_ci_low"
] = np.exp(
    np.minimum(
        item_plot[
            "expected_sign"
        ]
        * item_plot[
            "ci95_low"
        ],
        item_plot[
            "expected_sign"
        ]
        * item_plot[
            "ci95_high"
        ],
    )
)
item_plot[
    "oriented_ci_high"
] = np.exp(
    np.maximum(
        item_plot[
            "expected_sign"
        ]
        * item_plot[
            "ci95_low"
        ],
        item_plot[
            "expected_sign"
        ]
        * item_plot[
            "ci95_high"
        ],
    )
)
item_plot[
    "display_label"
] = (
    item_plot[
        "family"
    ]
    + "  "
    + item_plot[
        "feature_label"
    ]
    + " vs "
    + item_plot[
        "human_label"
    ]
)
item_plot = item_plot.sort_values(
    [
        "family",
        "pair_id",
    ]
).reset_index(
    drop=True
)

figure, axis = plt.subplots(
    figsize=(
        7.0,
        max(
            3.5,
            0.38
            * len(
                item_plot
            )
            + 1.2,
        ),
    ),
    constrained_layout=True,
)

for row_index, row in item_plot.iterrows():
    axis.errorbar(
        row[
            "oriented_or"
        ],
        row_index,
        xerr=[
            [
                row[
                    "oriented_or"
                ]
                - row[
                    "oriented_ci_low"
                ]
            ],
            [
                row[
                    "oriented_ci_high"
                ]
                - row[
                    "oriented_or"
                ]
            ],
        ],
        fmt="o",
        capsize=2,
    )

axis.axvline(
    1,
    linewidth=0.8,
)
axis.set_xscale(
    "log"
)
axis.set_yticks(
    np.arange(
        len(
            item_plot
        )
    )
)
axis.set_yticklabels(
    item_plot[
        "display_label"
    ]
)
axis.invert_yaxis()
axis.set_xlabel(
    (
        "Direction-oriented odds ratio "
        "(95% CI)"
    )
)
axis.set_title(
    (
        "Targeted item-level refinements "
        "(secondary analyses)"
    ),
    loc="left",
    fontweight="bold",
)
axis.grid(
    axis="x",
    alpha=0.25,
)

save_figure(
    figure,
    (
        "Supplementary_item_level_"
        "construct_refinements"
    ),
    main=False,
)
plt.show()


# Supplementary figure: leave-one-rater-out stability.
loo_figure_data = (
    leave_one_rater.loc[
        leave_one_rater[
            "model_status"
        ].eq("fit")
    ]
    .merge(
        CONSTRUCT_MAP[
            [
                "pair_id",
                "inference_tier",
                "main_display",
                "expected_sign",
            ]
        ],
        on="pair_id",
        how="left",
        validate="many_to_one",
    )
)

loo_figure_data = loo_figure_data.loc[
    loo_figure_data[
        "inference_tier"
    ].eq("primary_family")
    & loo_figure_data[
        "main_display"
    ]
].copy()

loo_figure_data[
    "oriented_or"
] = np.exp(
    loo_figure_data[
        "expected_sign"
    ]
    * np.log(
        loo_figure_data[
            "odds_ratio"
        ]
    )
)

loo_summary_plot = (
    loo_figure_data.groupby(
        [
            "pair_id",
            "family",
            "feature_label",
        ],
        as_index=False,
    )
    .agg(
        minimum_or=(
            "oriented_or",
            "min",
        ),
        maximum_or=(
            "oriented_or",
            "max",
        ),
    )
    .merge(
        forest_plot[
            [
                "pair_id",
                "oriented_or",
                "display_label",
            ]
        ],
        on="pair_id",
        how="inner",
        validate="one_to_one",
    )
)

loo_summary_plot = (
    loo_summary_plot.reset_index(
        drop=True
    )
)

figure, axis = plt.subplots(
    figsize=(
        7.0,
        max(
            4.0,
            0.34
            * len(
                loo_summary_plot
            )
            + 1.2,
        ),
    ),
    constrained_layout=True,
)

for row_index, row in (
    loo_summary_plot.iterrows()
):
    axis.plot(
        [
            row[
                "minimum_or"
            ],
            row[
                "maximum_or"
            ],
        ],
        [
            row_index,
            row_index,
        ],
        linewidth=1.2,
    )
    axis.scatter(
        row[
            "oriented_or"
        ],
        row_index,
        marker="o",
    )

axis.axvline(
    1,
    linewidth=0.8,
)
axis.set_xscale(
    "log"
)
axis.set_yticks(
    np.arange(
        len(
            loo_summary_plot
        )
    )
)
axis.set_yticklabels(
    loo_summary_plot[
        "display_label"
    ]
)
axis.invert_yaxis()
axis.set_xlabel(
    (
        "Full-model estimate and "
        "leave-one-rater-out range"
    )
)
axis.set_title(
    (
        "Sensitivity of family-level correspondence "
        "to omission of each rater"
    ),
    loc="left",
    fontweight="bold",
)
axis.grid(
    axis="x",
    alpha=0.25,
)

save_figure(
    figure,
    (
        "Supplementary_leave_one_rater_"
        "out_stability"
    ),
    main=False,
)
plt.show()


# Supplementary figure: sparse QDIST/QTEMP concordance.
sparse_plot = sparse_correspondence.copy()

figure, axis = plt.subplots(
    figsize=(6.4, 3.7),
    constrained_layout=True,
)

x = np.arange(
    len(sparse_plot)
)
width = 0.18

for offset, column, label in [
    (
        -1.5 * width,
        "both_positive_n11",
        "Both positive",
    ),
    (
        -0.5 * width,
        "machine_only_n10",
        "Machine only",
    ),
    (
        0.5 * width,
        "human_only_n01",
        "Human only",
    ),
    (
        1.5 * width,
        "both_negative_n00",
        "Both negative",
    ),
]:
    axis.bar(
        x + offset,
        sparse_plot[column],
        width=width,
        label=label,
    )

axis.set_xticks(
    x
)
axis.set_xticklabels(
    sparse_plot[
        "feature_label"
    ],
    rotation=20,
    ha="right",
)
axis.set_ylabel(
    "Recordings"
)
axis.set_title(
    "Sparse QDIST and QTEMP human–machine cross-tabulation",
    loc="left",
    fontweight="bold",
)
axis.legend(
    frameon=False,
    ncol=2,
)

save_figure(
    figure,
    "Supplementary_qdist_qtemp_sparse_correspondence",
    main=False,
)
plt.show()


## 20. Completion audit and manifest

In [ ]:
fit_primary = primary_models.loc[
    primary_models["model_status"].eq(
        "fit"
    )
]

support_failed_pairs = model_support.loc[
    ~model_support["support_pass"]
]

blocking_checks = pd.DataFrame(
    [
        {
            "check": "Frozen signal population reconstructed",
            "passed": (
                len(analysis_519)
                == EXPECTED_FROZEN_RECORDINGS
            ),
            "detail": (
                f"recordings={len(analysis_519)}"
            ),
        },
        {
            "check": "Frozen participant count reproduced",
            "passed": (
                analysis_519[
                    "participant_id"
                ].nunique()
                == EXPECTED_FROZEN_PARTICIPANTS
            ),
            "detail": (
                "participants="
                f"{analysis_519['participant_id'].nunique()}"
            ),
        },
        {
            "check": "Production human set reproduced",
            "passed": (
                len(
                    production_recording_union
                )
                == EXPECTED_HUMAN_RECORDINGS
            ),
            "detail": (
                "recordings="
                f"{len(production_recording_union)}"
            ),
        },
        {
            "check": "No production recording has multiple raters",
            "passed": all(
                frame[
                    "recording_id"
                ].is_unique
                for frame in human_outcome_tables.values()
            ),
            "detail": (
                "one production rating per recording"
            ),
        },
        {
            "check": "Four-rater strict-majority correction applied",
            "passed": (
                strict_consensus[
                    "strict_majority_at_least_three"
                ].notna().all()
            ),
            "detail": (
                "strict majority defined as at least 3 of 4"
            ),
        },
        {
            "check": "All supported primary models fit",
            "passed": (
                len(
                    fit_primary
                )
                == int(
                    model_support[
                        "support_pass"
                    ].sum()
                )
            ),
            "detail": (
                f"fit={len(fit_primary)}, "
                "supported="
                f"{int(model_support['support_pass'].sum())}"
            ),
        },
        {
            "check": "Participant-collapsed resampling success",
            "passed": (
                subject_collapsed_sensitivity[
                    "passes_success_gate"
                ].all()
                if len(
                    subject_collapsed_sensitivity
                )
                else False
            ),
            "detail": (
                "minimum success fraction="
                f"{subject_collapsed_sensitivity['success_fraction'].min():.3f}"
                if len(
                    subject_collapsed_sensitivity
                )
                else "no models"
            ),
        },
        {
            "check": "Cross-family specificity matrix complete",
            "passed": (
                len(specificity_fit)
                == len(SPECIFICITY_FEATURE_MAP)
                * len(SPECIFICITY_HUMAN_FAMILIES)
            ),
            "detail": (
                f"models={len(specificity_fit)}"
            ),
        },
        {
            "check": "Exact family-label specificity permutation complete",
            "passed": (
                len(specificity_permutations)
                == math.factorial(
                    len(
                        SPECIFICITY_HUMAN_FAMILIES
                    )
                )
            ),
            "detail": (
                "permutations="
                f"{len(specificity_permutations)}"
            ),
        },
        {
            "check": "QDIST and QTEMP kept descriptive",
            "passed": set(
                sparse_correspondence[
                    "family"
                ]
            )
            == {"QDIST", "QTEMP"},
            "detail": (
                "no inferential sparse-event models"
            ),
        },
        {
            "check": "No general human-machine temporal alignment",
            "passed": True,
            "detail": (
                "recording-level correspondence used for non-event families"
            ),
        },
    ]
)

display(blocking_checks)

export_dataframe(
    blocking_checks,
    LOG_DIR / "stage2_blocking_completion_checklist.csv",
)

manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "frozen_spec": FROZEN_SPEC,
    "frozen_recordings": len(
        analysis_519
    ),
    "frozen_participants": int(
        analysis_519[
            "participant_id"
        ].nunique()
    ),
    "production_human_recordings": len(
        production_recording_union
    ),
    "construct_pairs": len(
        CONSTRUCT_MAP
    ),
    "supported_construct_pairs": int(
        model_support[
            "support_pass"
        ].sum()
    ),
    "primary_models_fit": len(
        fit_primary
    ),
    "support_failed_pairs": (
        support_failed_pairs[
            "pair_id"
        ].tolist()
    ),
    "subject_collapsed_resamples": (
        N_SUBJECT_COLLAPSED_RESAMPLES
    ),
    "specificity_exact_permutation_p": (
        specificity_exact_p
    ),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "statsmodels": sm.__version__,
}

(
    MANIFEST_DIR / "stage2_run_manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

if not blocking_checks[
    "passed"
].all():
    print(
        "\nSTAGE 2 COMPLETED WITH BLOCKING ITEMS. "
        "Do not use the figure or manuscript results until reviewed."
    )
else:
    print(
        "\nSTAGE 2 COMPLETED. "
        "The construct-matched human correspondence outputs are ready for review."
    )


---

## Interpretation boundary

A supported association indicates that a signal-derived indicator covaries with a trained rater’s construct-matched family label after accounting for participant clustering, rater, cohort, and recording duration. It does not make the human label ground truth, and it does not establish a physical source.

The main manuscript result is family-level and indicator-specific. Item-level analyses only refine narrow subconstructs. Rater-specific analyses are diagnostics and remain supplementary. QDIST and QTEMP remain descriptive because their machine events are sparse and their positive perceptual labels are weakly reproducible or rare.
